# Nighttime Thermal Environments Are Associated with Impaired Sleep Among Outdoor Workers in Non-Air-Conditioned Dormitories

- **Project:** HEATS — Cooling Dorms
- **Sites:** Dormitory A & Dormitory B, Singapore
- **Author:** Raagavi Mani, 2026
- **Data:** all source files used by this notebook are shared on Zenodo [10.5281/zenodo.22821941]

## This notebook reproduces every analysis reported in the Methods and Results sections of the manuscript, in the order the manuscript describes them:

1. **Environmental data** — load Dormitory A & B sensor logs, derive absolute humidity,
   wet bulb temperature.
2. **Figures 1 and S1** — 24-hour indoor/outdoor profiles and nighttime boxplots for
   temperature, humidity, wet bulb temperature, CO₂ and PM₂.₅.
3. **Table 1 / Tables S2–S4** — descriptive statistics of the indoor dormitory environment during nighttime sleep periods.
4. **Sleep data pipeline** — actigraphy → subjective-survey merge → environmental-exposure
   attachment → data-cleaning/exclusion cascade, producing the final analytic sample of **694 nights from
   38 participants**.
5. **Assumption checks** — normality, threshold exceedance, collinearity/VIF among
   exposures
6. **Table S1 / Figure 2** — within-subject OLS regressions of sleep outcomes on each
   environmental exposure, with subject-clustered SEs, room fixed effects and Bonferroni
   correction.
7. **Figure 3** — thermal sensation, thermal preference and air movement preference ratings.

## 1. Setup

In [1]:
import os
from datetime import datetime

# ----------------------------------------------------------------------------
# Source data ships in the "data/" folder next to this notebook and is never
# written to. Everything this notebook generates -- intermediate pipeline
# CSVs, figures, tables -- goes under "outputs/" instead.
# ----------------------------------------------------------------------------
DATA_DIR = "data"
OUTPUT_DATA_DIR = os.path.join("outputs", "data")
FIGURES_DIR = os.path.join("outputs", "figures")
TABLES_DIR = os.path.join("outputs", "tables")
os.makedirs(OUTPUT_DATA_DIR, exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)
os.makedirs(TABLES_DIR, exist_ok=True)

RUN_STARTED_AT = datetime.now().strftime('%d %b %Y, %H:%M:%S')
print(f"Notebook run started {RUN_STARTED_AT}")

Notebook run started 18 Sep 2026, 09:18:15


## 2. Environmental Data — Loading & Derived Metrics

Indoor and outdoor temperature, relative humidity, CO₂ and PM₂.₅ were logged at 1-minute intervals by Atmocube monitors. From calibrated temperature and relative humidity we derive:

- **Wet bulb temperature** (Stull's formula)
- **Absolute humidity** — vapor pressure from `pythermalcomfort`'s `psy_ta_rh`, converted to
  g·m⁻³ via the ideal-gas law (`abs_humidity_c`). This is the only absolute-humidity value
  used anywhere downstream: the raw on-device `abs_humidity` estimate that Atmocube also
  ships is dropped immediately below and never enters any table, plot or regression in this
  notebook.

In [2]:
import pandas as pd
from datetime import datetime, timedelta
import numpy as np
from pythermalcomfort.utilities import psy_ta_rh

# Folder containing file
dorm_A_env_file_path = os.path.join(DATA_DIR, "env_dorm_A.parquet.gzip")
dorm_B_env_file_path = os.path.join(DATA_DIR, "env_dorm_B.parquet.gzip")

# Define time range
dorm_A_start = "2025-04-26 18:00:00"
dorm_A_end = "2025-05-31 10:00:00"

dorm_B_phase1_start = "2025-09-27 18:00:00"
dorm_B_phase1_end = "2025-10-18 12:00:00"

dorm_B_phase2_start = "2025-10-25 18:00:00"
dorm_B_phase2_end = "2025-11-15 12:00:00"

# File path
dorm_A_df = pd.read_parquet(dorm_A_env_file_path, engine="pyarrow")
dorm_B_df = pd.read_parquet(dorm_B_env_file_path, engine="pyarrow")

# Calculate wet bulb temperature and add as a new column in-place

TEMP_COL = "temperature_calibrated"
RH_COL = "humidity_calibrated"
OUTPUT_COL = "wet_bulb_temp"

def calculate_wet_bulb(T, RH):
    """
    Calculates wet bulb temperature using the Stull formula.

    Parameters
    ----------
    T : array-like
        Air temperature in Celsius
    RH : array-like
        Relative humidity in %

    Returns
    -------
    np.ndarray
        Wet bulb temperature in Celsius
    """

    T = np.asarray(T, dtype=float)
    RH = np.asarray(RH, dtype=float)

    term1 = T * np.arctan(0.151977 * np.sqrt(RH + 8.313659))
    term2 = np.arctan(T + RH)
    term3 = np.arctan(RH - 1.676331)
    term4 = 0.00391838 * (RH ** 1.5) * np.arctan(0.023101 * RH)

    wet_bulb = term1 + term2 - term3 + term4 - 4.686035

    return wet_bulb

def add_wet_bulb_column_inplace(
    df,
    temp_col=TEMP_COL,
    rh_col=RH_COL,
    output_col=OUTPUT_COL
):
    df[output_col] = np.nan

    missing_cols = [c for c in [temp_col, rh_col] if c not in df.columns]
    if missing_cols:
        raise KeyError(f"Missing required columns: {missing_cols}")

    valid = df[temp_col].notna() & df[rh_col].notna()

    if valid.any():
        df.loc[valid, output_col] = calculate_wet_bulb(
            T=df.loc[valid, temp_col].astype(float).to_numpy(),
            RH=df.loc[valid, rh_col].astype(float).to_numpy()
        )

add_wet_bulb_column_inplace(dorm_A_df)
add_wet_bulb_column_inplace(dorm_B_df)

# Calculate abs_humidity_c and add as a new column in-place
TEMP_COL = "temperature_calibrated"
RH_COL = "humidity_calibrated"
OUTPUT_COL = "abs_humidity_c"

# ── Absolute humidity from calibrated T & RH (pythermalcomfort) ───────────────

def calculate_abs_humidity_c(T, RH):
    """
    Volumetric absolute humidity (g·m⁻³) from dry-bulb air temperature (°C)
    and relative humidity (%).

    The water-vapour partial pressure e is obtained from pythermalcomfort's
    `psy_ta_rh`; absolute humidity then follows from the ideal-gas law
    ρ_v = M_w·e / (R·T), i.e. AH[g·m⁻³] = 2.1668·e[Pa] / T[K]."""
    T = np.asarray(T, dtype="float64")
    RH  = np.asarray(RH,  dtype="float64")
    abs_humidity_c  = np.full(np.broadcast(T, RH).shape, np.nan, dtype="float64")
    tdb_b, rh_b = np.broadcast_arrays(T, RH)
    ok  = np.isfinite(tdb_b) & np.isfinite(rh_b)
    if ok.any():
        e_pa = np.asarray(
            psy_ta_rh(tdb=tdb_b[ok], rh=np.clip(rh_b[ok], 0.0, 100.0)).p_vap,
            dtype="float64")
        abs_humidity_c[ok] = 2.1668 * e_pa / (tdb_b[ok] + 273.15)
    return abs_humidity_c


def add_abs_humidity_column_inplace(
    df,
    temp_col=TEMP_COL,
    rh_col=RH_COL,
    output_col=OUTPUT_COL
):
    df[output_col] = np.nan

    missing_cols = [c for c in [temp_col, rh_col] if c not in df.columns]
    if missing_cols:
        raise KeyError(f"Missing required columns: {missing_cols}")

    valid = df[temp_col].notna() & df[rh_col].notna()

    if valid.any():
        df.loc[valid, output_col] = calculate_abs_humidity_c(
            T=df.loc[valid, temp_col].astype(float).to_numpy(),
            RH=df.loc[valid, rh_col].astype(float).to_numpy()
        )


add_abs_humidity_column_inplace(dorm_A_df)
add_abs_humidity_column_inplace(dorm_B_df)

# Convert index to datetime if it's not already
dorm_A_df.index = pd.to_datetime(dorm_A_df.index)
dorm_B_df.index = pd.to_datetime(dorm_B_df.index)


def ensure_co2_used(df: pd.DataFrame) -> pd.DataFrame:
    """Guarantee a 'co2_used' column exists (co2_corrected where available,
    else raw co2). Defined once here and reused everywhere below instead of
    being re-implemented per section."""
    df = df.copy()
    if "co2_used" in df.columns:
        return df
    if "co2_corrected" in df.columns:
        df["co2_used"] = df["co2_corrected"]
    elif "co2" in df.columns:
        df["co2_used"] = df["co2"]
    else:
        raise KeyError("Neither 'co2_used', 'co2_corrected' nor 'co2' found")
    return df


# ----------------------------------------------------------------------------
# The shared data files (env_dorm_A.parquet.gzip / env_dorm_B.parquet.gzip)
# are already restricted to the study's control ("CON") arm rooms, phases and
# date range at the source: intervention-arm rows, INT-only rooms, and the
# pre-indoor/outdoor-split rows have already been removed before
# these files were shared. The date-window filter below is therefore a no-op
# on the shared files (kept only for documentation of the study period); what
# actually matters is the assertion right after it, which fails loudly if a
# future data drop ever contains an unexpected room instead of silently
# mixing intervention-arm data back in.
# ----------------------------------------------------------------------------
dorm_A_condata_df = dorm_A_df[
    (dorm_A_df.index >= dorm_A_start) & (dorm_A_df.index <= dorm_A_end)
].copy()
dorm_A_condata_df = ensure_co2_used(dorm_A_condata_df)

EXPECTED_DORM_A_ROOMS = {"B205", "B206", "B207", "Outside 1"}
actual_dorm_A_rooms = set(dorm_A_condata_df["id_room"].unique())
assert actual_dorm_A_rooms == EXPECTED_DORM_A_ROOMS, (
    f"\u274c Unexpected Dorm A rooms in env_dorm_A.parquet.gzip: {actual_dorm_A_rooms}. "
    f"Expected exactly the control-arm rooms {EXPECTED_DORM_A_ROOMS} -- has the shared "
    f"file changed?"
)

dorm_B_df_phase1_condata_df = dorm_B_df[
    (dorm_B_df.index >= dorm_B_phase1_start) & (dorm_B_df.index <= dorm_B_phase1_end)
].copy()
dorm_B_df_phase2_condata_df = dorm_B_df[
    (dorm_B_df.index >= dorm_B_phase2_start) & (dorm_B_df.index <= dorm_B_phase2_end)
].copy()

EXPECTED_DORM_B_PHASE1_ROOMS = {"A104 Indoor", "A104 Outdoor", "A305 Indoor", "A305 Outdoor"}
EXPECTED_DORM_B_PHASE2_ROOMS = {
    "B102 Indoor", "B102 Outdoor", "A201 Indoor", "A201 Outdoor", "B302 Indoor", "B302 Outdoor",
}
actual_dorm_B_p1_rooms = set(dorm_B_df_phase1_condata_df["id_room"].unique())
actual_dorm_B_p2_rooms = set(dorm_B_df_phase2_condata_df["id_room"].unique())
assert actual_dorm_B_p1_rooms == EXPECTED_DORM_B_PHASE1_ROOMS, (
    f"\u274c Unexpected Dorm B phase-1 rooms: {actual_dorm_B_p1_rooms}. "
    f"Expected {EXPECTED_DORM_B_PHASE1_ROOMS}."
)
assert actual_dorm_B_p2_rooms == EXPECTED_DORM_B_PHASE2_ROOMS, (
    f"\u274c Unexpected Dorm B phase-2 rooms: {actual_dorm_B_p2_rooms}. "
    f"Expected {EXPECTED_DORM_B_PHASE2_ROOMS}."
)

# One combined Dorm B control-arm frame, built once here and reused by every
# later section (Figure 1/S1, Section 6's descriptive tables, and the sleep
# exposure-attachment step) instead of being rebuilt -- and re-broken -- per
# section.
dorm_B_condata_df = pd.concat(
    [dorm_B_df_phase1_condata_df, dorm_B_df_phase2_condata_df]
).sort_index()
dorm_B_condata_df = ensure_co2_used(dorm_B_condata_df)

# Kept under their original names for the sleep-exposure attachment step
# (Section 7.3) further down the notebook.
dorm_A_condata_df0 = dorm_A_condata_df.copy()
dorm_B_condata_df0 = dorm_B_condata_df.copy()

print("\u2b50 Dorm A control-arm rooms:", sorted(actual_dorm_A_rooms))
print("\u2b50 Dorm B control-arm rooms \u2014 phase 1:", sorted(actual_dorm_B_p1_rooms))
print("\u2b50 Dorm B control-arm rooms \u2014 phase 2:", sorted(actual_dorm_B_p2_rooms))
print(f"Dorm A control-arm rows: {len(dorm_A_condata_df)}")
print(f"Dorm B control-arm rows: {len(dorm_B_condata_df)}")

⭐ Dorm A control-arm rooms: ['B205', 'B206', 'B207', 'Outside 1']
⭐ Dorm B control-arm rooms — phase 1: ['A104 Indoor', 'A104 Outdoor', 'A305 Indoor', 'A305 Outdoor']
⭐ Dorm B control-arm rooms — phase 2: ['A201 Indoor', 'A201 Outdoor', 'B102 Indoor', 'B102 Outdoor', 'B302 Indoor', 'B302 Outdoor']
Dorm A control-arm rows: 203685
Dorm B control-arm rows: 267504


Dormitory B was profiled in two date-bounded phases (rooms were swapped between the
control arm and the other arm between phases); the shared data file already reflects each
room's correct phase, so the two phases' control-arm rows are simply concatenated into one
`dorm_B_condata_df` here -- built once, and reused by every later section instead of being
rebuilt per section.

In [3]:
# Control-arm room -> participant assignments (used in Section 7.1 to
# restrict the actigraphy sleep-metrics file to control-arm participants).
CON_ROOM_ASSIGNMENTS = {
    'B205' : ['CDS025', 'CDS026', 'CDS027'], #3 pax
    'B206' : ['CDS019', 'CDS029'], #2 pax
    'B207' : ['CDS030', 'CDS032', 'CDS035', 'CDS037', 'CDS038', 'CDS040'], #6 pax
    'A104': ['CDS115', 'CDS116', 'CDS117', 'CDS118', 'CDS119', 'CDS120', 'CDS121', 'CDS122', 'CDS123'],
    'A305': ['CDS124', 'CDS126'],
    'B102': ['CDS101', 'CDS102', 'CDS103', 'CDS128', 'CDS129', 'CDS130', 'CDS131', 'CDS132'],
    'A201': ['CDS104', 'CDS105'],
    'B302': ['CDS106', 'CDS107', 'CDS108', 'CDS109', 'CDS110', 'CDS111', 'CDS112', 'CDS113', 'CDS114'],
}

print(f"Control-arm rooms: {list(CON_ROOM_ASSIGNMENTS.keys())}")
print(f"Control-arm participants: {sum(len(v) for v in CON_ROOM_ASSIGNMENTS.values())}")

Control-arm rooms: ['B205', 'B206', 'B207', 'A104', 'A305', 'B102', 'A201', 'B302']
Control-arm participants: 41


## 3. Indoor/Outdoor Dataset Construction

The indoor vs. outdoor sensor split, the noon-anchored hour index used for the 24-hour
profile plots, and the exclusion of one faulty-sensor night in Dormitory A are all shared by
every environmental variable plotted below (temperature, humidity, wet-bulb temperature,
CO₂, PM₂.₅). They are built once here rather than being repeated per variable.

In [4]:
def add_hour_from_noon(df: pd.DataFrame) -> pd.DataFrame:
    """Hour-of-day index running 0-23 starting at noon (12:00 -> 0, 11:00 -> 23),
    so a 24h profile can be plotted noon-to-noon with nighttime centered."""
    df = df.copy()
    df["id_room"] = df["id_room"].astype(str)
    df["hour"] = df.index.hour
    df["hour_from_noon"] = ((df["hour"] - 12) % 24).astype(int)
    return df

DORM_A_INDOOR_ROOMS = ["B205", "B206", "B207"]
DORM_A_OUTDOOR_ROOM = "Outside 1" 

DORM_B_ROOM_LABELS = {
    "A104 Indoor": "Room A104",
    "A305 Indoor": "Room A305",
    "B102 Indoor": "Room B102",
    "A201 Indoor": "Room A201",
    "B302 Indoor": "Room B302",
}
DORM_B_INDOOR_ROOMS = list(DORM_B_ROOM_LABELS.keys())
DORM_B_OUTDOOR_ROOMS = [r.replace("Indoor", "Outdoor") for r in DORM_B_INDOOR_ROOMS]

# ---- Dorm A: split indoor vs. outdoor, drop one night with a known sensor fault ----
dorm_A = add_hour_from_noon(dorm_A_condata_df)
_faulty_night = (
    (dorm_A.index >= pd.Timestamp("2025-05-06 19:00", tz="Asia/Singapore")) &
    (dorm_A.index < pd.Timestamp("2025-05-07 07:00", tz="Asia/Singapore"))
)
dorm_A_f = dorm_A[
    dorm_A["id_room"].isin(DORM_A_INDOOR_ROOMS + [DORM_A_OUTDOOR_ROOM]) & ~_faulty_night
].copy()

dorm_A_indoor_raw = dorm_A_f[dorm_A_f["id_room"].isin(DORM_A_INDOOR_ROOMS)].copy()
dorm_A_outdoor_raw = dorm_A_f[dorm_A_f["id_room"] == DORM_A_OUTDOOR_ROOM].copy()

# ---- Dorm B: already built once, control-arm only, in Section 2 above ----
dorm_B = add_hour_from_noon(dorm_B_condata_df)

dorm_B_f = dorm_B[dorm_B["id_room"].isin(DORM_B_INDOOR_ROOMS + DORM_B_OUTDOOR_ROOMS)].copy()
dorm_B_f["io"] = dorm_B_f["id_room"].apply(
    lambda s: "Indoor" if "Indoor" in s else ("Outdoor" if "Outdoor" in s else "Unknown")
)

dorm_B_indoor_raw = dorm_B_f[dorm_B_f["io"] == "Indoor"].copy()
dorm_B_outdoor_raw = dorm_B_f[dorm_B_f["io"] == "Outdoor"].copy()

print(f"Dorm A — indoor rows: {len(dorm_A_indoor_raw)}, outdoor rows: {len(dorm_A_outdoor_raw)}")
print(f"Dorm B — indoor rows: {len(dorm_B_indoor_raw)}, outdoor rows: {len(dorm_B_outdoor_raw)}")

Dorm A — indoor rows: 155976, outdoor rows: 46773
Dorm B — indoor rows: 140079, outdoor rows: 127425


## 4. Reusable Plotting Helpers

Every panel of Figure 1 and Figure S1 is one of two chart types — a 24-hour indoor/outdoor
mean profile, or a nighttime indoor/outdoor boxplot — repeated across six environmental
variables. `plot_profile()` and `plot_boxplot()` implement each chart type once; the actual
per-variable cells below are then a few lines each.

In [5]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats

ALPHA = 0.05
MIN_N_PER_GROUP = 5  # require at least this many samples per hour per group to test

ORANGE = "#FFA500"   # Indoor
BLUE = "#0072B2"      # Outdoor
BLACK = "black"
DARK_BLUE = "#021923"

TITLE_FONT_SIZE = 32
AXIS_TITLE_SIZE = 30
TICK_FONT_SIZE = 26
LEGEND_FONT_SIZE = 26
NIGHTTIME_FONT_SIZE = 26

SHOW_SD_BANDS = True
SD_BAND_OPACITY = 0.18

# 24h profile x-axis: noon -> 11am, labelled every 3 hours
TICKVALS_NOON = [0, 3, 6, 9, 12, 15, 18, 21]
TICKTEXT_NOON = ["12", "15", "18", "21", "0", "3", "6", "9"]


# ---- shared helpers: 24h profile ----
def get_hourly_samples(df, value_col, hour_col="hour_from_noon"):
    """Per-hour arrays of non-null values (0-23), for per-hour significance testing."""
    return {h: df.loc[df[hour_col] == h, value_col].dropna().astype(float).values for h in range(24)}


def hourly_mean_line(df, value_col, hour_col="hour_from_noon"):
    return (
        df[[hour_col, value_col]]
        .dropna(subset=[value_col])
        .groupby(hour_col)[value_col]
        .agg(mean="mean", sd="std")
        .reindex(range(24))
        .reset_index()
        .rename(columns={hour_col: "hour_from_noon", "mean": value_col})
    )


def pooled_hourly_line(df1, df2, value_col, hour_col="hour_from_noon"):
    """One overall hourly mean/SD line pooled across every reading from both
    dormitories, so each monitored room contributes in proportion to the data it
    actually recorded -- matching the figure caption ("hourly means aggregated
    across all monitored rooms").

    This replaced an earlier version that averaged Dorm A's and Dorm B's
    dormitory-level hourly means with equal weight, and averaged the two
    dormitories' SDs (which is not a pooled SD). The band drawn from the `sd`
    column here is a genuine pooled SD across all rooms."""
    pooled = pd.concat(
        [df1[[hour_col, value_col]], df2[[hour_col, value_col]]],
        ignore_index=True,
    )
    return hourly_mean_line(pooled, value_col, hour_col)


def welch_sig_hours(samples_a, samples_b, alpha=ALPHA, min_n=MIN_N_PER_GROUP):
    """Per-hour Welch t-test between Dorm A and Dorm B (diagnostic only; not plotted)."""
    sig_hours, pvals = [], {}
    for h in range(24):
        a, b = samples_a[h], samples_b[h]
        if len(a) >= min_n and len(b) >= min_n:
            _, p = stats.ttest_ind(a, b, equal_var=False, nan_policy="omit")
            pvals[h] = p
            if p < alpha:
                sig_hours.append(h)
        else:
            pvals[h] = np.nan
    return sig_hours, pvals


def add_sd_band(fig, df, color, name, value_col):
    band_df = df.dropna(subset=[value_col, "sd"]).copy()
    x = band_df["hour_from_noon"]
    y_upper = band_df[value_col] + band_df["sd"]
    y_lower = band_df[value_col] - band_df["sd"]
    fig.add_trace(go.Scatter(
        x=pd.concat([x, x[::-1]]), y=pd.concat([y_upper, y_lower[::-1]]),
        fill="toself", fillcolor=color, line=dict(color="rgba(255,255,255,0)"),
        hoverinfo="skip", showlegend=False, opacity=SD_BAND_OPACITY, name=f"{name} ± SD",
    ))


def plot_profile(poi, y_axis_title, y_range=None, y_dtick=None, filename=None, label=None):
    """24-hour mean indoor vs outdoor profile (noon-to-noon), Dorm A + Dorm B combined,
    with nighttime (19:00-07:00) shading. Prints per-hour Dorm A vs Dorm B significance
    (Welch t-test) as a diagnostic; this is not annotated on the figure itself."""
    label = label or poi
    indoor_a_samples = get_hourly_samples(dorm_A_indoor_raw, poi)
    indoor_b_samples = get_hourly_samples(dorm_B_indoor_raw, poi)
    outdoor_a_samples = get_hourly_samples(dorm_A_outdoor_raw, poi)
    outdoor_b_samples = get_hourly_samples(dorm_B_outdoor_raw, poi)

    indoor_overall = pooled_hourly_line(dorm_A_indoor_raw, dorm_B_indoor_raw, poi)
    outdoor_overall = pooled_hourly_line(dorm_A_outdoor_raw, dorm_B_outdoor_raw, poi)

    sig_hours_indoor, _ = welch_sig_hours(indoor_a_samples, indoor_b_samples)
    sig_hours_outdoor, _ = welch_sig_hours(outdoor_a_samples, outdoor_b_samples)
    print(f"[{label}] Dorm A vs Dorm B significant hours — Indoor: {sig_hours_indoor}")
    print(f"[{label}] Dorm A vs Dorm B significant hours — Outdoor: {sig_hours_outdoor}")

    fig = go.Figure()
    if SHOW_SD_BANDS:
        add_sd_band(fig, outdoor_overall, BLUE, "Outdoor", poi)
        add_sd_band(fig, indoor_overall, ORANGE, "Indoor", poi)

    fig.add_trace(go.Scatter(
        x=outdoor_overall["hour_from_noon"], y=outdoor_overall[poi], mode="lines+markers",
        name="Outdoor", line=dict(color=BLUE, width=6), marker=dict(size=11, symbol="circle"),
    ))
    fig.add_trace(go.Scatter(
        x=indoor_overall["hour_from_noon"], y=indoor_overall[poi], mode="lines+markers",
        name="Indoor", line=dict(color=ORANGE, width=6), marker=dict(size=11, symbol="circle"),
    ))

    fig.update_layout(
        template="simple_white", height=650, width=1400, margin=dict(r=260),
        font=dict(family="Arial", size=17),
        legend=dict(x=0.96, y=1, xanchor="left", yanchor="top",
                    font=dict(size=LEGEND_FONT_SIZE, family="Arial", color="black")),
    )
    fig.update_xaxes(
        title=dict(text="Hour of Day", font=dict(size=AXIS_TITLE_SIZE, family="Arial", color="black")),
        type="linear", tickmode="array", tickvals=TICKVALS_NOON, ticktext=TICKTEXT_NOON, dtick=1,
        tickfont=dict(size=TICK_FONT_SIZE, family="Arial", color="black"),
        showgrid=False, showline=True, linewidth=3, linecolor="black",
        ticks="outside", ticklen=6, tickwidth=1,
    )
    fig.update_yaxes(
        title=dict(text=y_axis_title, font=dict(size=AXIS_TITLE_SIZE, family="Arial", color="black")),
        tickfont=dict(size=TICK_FONT_SIZE, family="Arial", color="black"),
        showgrid=False, showline=True, linewidth=3, linecolor="black",
        ticks="outside", ticklen=6, tickwidth=1, range=y_range, dtick=y_dtick,
    )
    fig.add_vrect(x0=7, x1=19, fillcolor="rgba(0, 0, 0, 0.2)", layer="below", line_width=0)
    fig.add_annotation(
        x=13, y=1.01, xref="x", yref="paper", text="Nighttime", showarrow=False,
        font=dict(size=NIGHTTIME_FONT_SIZE, color="grey", family="Arial"),
        xanchor="center", yanchor="bottom",
    )

    fig.show()
    if filename:
        fig.write_image(os.path.join(FIGURES_DIR, filename), width=1400, height=650, scale=1)
        # fig.write_image(os.path.join(FIGURES_DIR, filename.rsplit(".", 1)[0] + ".tiff"), width=1400, height=650, scale=6, format="tiff")  # 600 dpi TIFF (uncomment to export)
    # No return value: fig.show() above already renders the figure. Returning fig here would
    # cause it to render a second time whenever this call is the last statement in a cell,
    # since Jupyter auto-displays an unassigned trailing expression.


# ---- shared helpers: nighttime boxplot ----
def filter_nighttime(df, value_col, hour_col="hour_from_noon"):
    """Keep 19:00-06:59 only (hour_from_noon in [7, 19))."""
    out = df[(df[hour_col] >= 7) & (df[hour_col] < 19)].copy()
    return out.dropna(subset=[value_col])


def add_night_date(df):
    """Anchor each row to the calendar date its night started (19:00-23:59 keeps that
    date; 00:00-06:59 belongs to the previous night)."""
    out = df.copy()
    ts = pd.Series(out.index.tz_localize(None), index=out.index)
    night_date = ts.dt.normalize()
    night_date = night_date.where(ts.dt.hour >= 7, night_date - pd.Timedelta(days=1))
    out["night_date"] = night_date.values
    return out


def count_nights(df):
    if "night_date" not in df.columns:
        df = add_night_date(df)
    return df["night_date"].nunique()


def box_stats(df, value_col):
    vals = df[value_col].dropna().astype(float)
    return {
        "n_samples": len(vals), "min": vals.min(), "q1": vals.quantile(0.25),
        "median": vals.quantile(0.50), "q3": vals.quantile(0.75), "max": vals.max(),
    }


def _nighttime_indoor_outdoor(poi):
    dorm_A_indoor_night = add_night_date(filter_nighttime(dorm_A_indoor_raw, poi))
    dorm_A_outdoor_night = add_night_date(filter_nighttime(dorm_A_outdoor_raw, poi))
    dorm_B_indoor_night = add_night_date(filter_nighttime(dorm_B_indoor_raw, poi))
    dorm_B_outdoor_night = add_night_date(filter_nighttime(dorm_B_outdoor_raw, poi))

    indoor_night = pd.concat([
        dorm_A_indoor_night[[poi, "night_date"]], dorm_B_indoor_night[[poi, "night_date"]]
    ])
    outdoor_night = pd.concat([
        dorm_A_outdoor_night[[poi, "night_date"]], dorm_B_outdoor_night[[poi, "night_date"]]
    ])
    night_counts = dict(
        dorm_A_indoor=count_nights(dorm_A_indoor_night), dorm_A_outdoor=count_nights(dorm_A_outdoor_night),
        dorm_B_indoor=count_nights(dorm_B_indoor_night), dorm_B_outdoor=count_nights(dorm_B_outdoor_night),
        overall_indoor=count_nights(indoor_night), overall_outdoor=count_nights(outdoor_night),
    )
    return indoor_night, outdoor_night, night_counts


def plot_boxplot(poi, y_axis_title, label_decimals=1, filename=None, label=None):
    """Nighttime (19:00-07:00) indoor vs outdoor boxplot, Dorm A + Dorm B nights combined."""
    label = label or poi
    indoor_night, outdoor_night, night_counts = _nighttime_indoor_outdoor(poi)
    print(f"[{label}] Nights used: {night_counts}")

    indoor_stats = box_stats(indoor_night, poi)
    outdoor_stats = box_stats(outdoor_night, poi)
    print(f"[{label}] Indoor nighttime stats: {indoor_stats}")
    print(f"[{label}] Outdoor nighttime stats: {outdoor_stats}")

    fig_box = go.Figure()
    fig_box.add_trace(go.Box(
        x=[0], name="Outdoor", q1=[outdoor_stats["q1"]], median=[outdoor_stats["median"]],
        q3=[outdoor_stats["q3"]], lowerfence=[outdoor_stats["min"]], upperfence=[outdoor_stats["max"]],
        line=dict(color=DARK_BLUE, width=3), fillcolor=BLUE, opacity=0.9, boxpoints=False,
        showlegend=False, width=0.45,
    ))
    fig_box.add_trace(go.Box(
        x=[1], name="Indoor", q1=[indoor_stats["q1"]], median=[indoor_stats["median"]],
        q3=[indoor_stats["q3"]], lowerfence=[indoor_stats["min"]], upperfence=[indoor_stats["max"]],
        line=dict(color=BLACK, width=3), fillcolor=ORANGE, opacity=0.9, boxpoints=False,
        showlegend=False, width=0.45,
    ))
    fig_box.add_trace(go.Scatter(
        x=[0.28, 1.28], y=[outdoor_stats["median"], indoor_stats["median"]], mode="text",
        text=[f'{outdoor_stats["median"]:.{label_decimals}f}', f'{indoor_stats["median"]:.{label_decimals}f}'],
        textposition="middle right", textfont=dict(size=TICK_FONT_SIZE, color="black"), showlegend=False,
    ))

    all_vals = pd.concat([indoor_night[poi], outdoor_night[poi]]).dropna().astype(float)
    ymin, ymax = np.floor(all_vals.min() - 0.5), np.ceil(all_vals.max() + 0.5)

    fig_box.update_layout(
        template="plotly_white", height=650, width=800, margin=dict(r=40, l=90, t=100, b=80),
        font=dict(family="Arial", size=17, color=BLACK),
        legend=dict(x=1.2, y=0.98, xanchor="right", yanchor="top",
                    font=dict(family="Arial", size=LEGEND_FONT_SIZE, color=BLACK), bgcolor="rgba(255,255,255,0)"),
        boxmode="group",
    )
    fig_box.update_xaxes(
        title=dict(text="Setting", font=dict(size=AXIS_TITLE_SIZE, family="Arial", color=BLACK)),
        tickmode="array", tickvals=[0, 1], ticktext=["Outdoor", "Indoor"], range=[-0.5, 1.5],
        tickfont=dict(size=TICK_FONT_SIZE, family="Arial", color=BLACK),
        showgrid=False, showline=True, linewidth=3, linecolor=BLACK, ticks="outside", ticklen=6, tickwidth=1,
    )
    fig_box.update_yaxes(
        title=dict(text=y_axis_title, font=dict(size=AXIS_TITLE_SIZE, family="Arial", color=BLACK)),
        tickfont=dict(size=TICK_FONT_SIZE, family="Arial", color=BLACK),
        showgrid=False, showline=True, linewidth=3, linecolor=BLACK, ticks="outside", ticklen=6, tickwidth=1,
        range=[ymin, ymax],
    )

    fig_box.show()
    if filename:
        fig_box.write_image(os.path.join(FIGURES_DIR, filename), width=800, height=650, scale=1)
        # fig_box.write_image(os.path.join(FIGURES_DIR, filename.rsplit(".", 1)[0] + ".tiff"), width=800, height=650, scale=6, format="tiff")  # 600 dpi TIFF (uncomment to export)
    # No return value -- see the note at the end of plot_profile() above.


def plot_boxplot_broken_axis(poi, y_axis_title, lower_range, upper_range, lower_tickvals,
                              upper_tickvals, label_decimals=0, filename=None, label=None):
    """Nighttime indoor vs outdoor boxplot with a broken y-axis, for variables (PM2.5) whose
    outdoor readings include occasional large spikes that would otherwise dwarf the boxes."""
    label = label or poi
    indoor_night, outdoor_night, night_counts = _nighttime_indoor_outdoor(poi)
    print(f"[{label}] Nights used: {night_counts}")

    indoor_stats = box_stats(indoor_night, poi)
    outdoor_stats = box_stats(outdoor_night, poi)
    print(f"[{label}] Indoor nighttime stats: {indoor_stats}")
    print(f"[{label}] Outdoor nighttime stats: {outdoor_stats}")

    def add_box(fig, s, x, name, fillcolor, linecolor, row):
        fig.add_trace(go.Box(
            x=[x], name=name, q1=[s["q1"]], median=[s["median"]], q3=[s["q3"]],
            lowerfence=[s["min"]], upperfence=[s["max"]], line=dict(color=linecolor, width=3),
            fillcolor=fillcolor, opacity=0.9, boxpoints=False, showlegend=False, width=0.45,
        ), row=row, col=1)

    fig_box = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
                             row_heights=[0.30, 0.70])
    add_box(fig_box, outdoor_stats, 0, "Outdoor", BLUE, DARK_BLUE, row=1)
    add_box(fig_box, indoor_stats, 1, "Indoor", ORANGE, BLACK, row=1)
    add_box(fig_box, outdoor_stats, 0, "Outdoor", BLUE, DARK_BLUE, row=2)
    add_box(fig_box, indoor_stats, 1, "Indoor", ORANGE, BLACK, row=2)

    fig_box.add_trace(go.Scatter(
        x=[0.28, 1.28], y=[outdoor_stats["median"], indoor_stats["median"]], mode="text",
        text=[f'{outdoor_stats["median"]:.{label_decimals}f}', f'{indoor_stats["median"]:.{label_decimals}f}'],
        textposition="middle right", textfont=dict(size=TICK_FONT_SIZE, color=BLACK, family="Arial"),
        showlegend=False, hoverinfo="skip",
    ), row=2, col=1)

    fig_box.update_layout(
        template="plotly_white", height=650, width=800, margin=dict(r=40, l=130, t=100, b=80),
        font=dict(family="Arial", size=17, color=BLACK), boxmode="group",
    )
    fig_box.update_xaxes(showticklabels=False, title_text="", showgrid=False, showline=False,
                          linewidth=3, linecolor=BLACK, ticks="", row=1, col=1)
    fig_box.update_xaxes(
        title=dict(text="Setting", font=dict(size=AXIS_TITLE_SIZE, family="Arial", color=BLACK)),
        tickmode="array", tickvals=[0, 1], ticktext=["Outdoor", "Indoor"], range=[-0.5, 1.5],
        tickfont=dict(size=TICK_FONT_SIZE, family="Arial", color=BLACK),
        showgrid=False, showline=True, linewidth=3, linecolor=BLACK, ticks="outside",
        ticklen=6, tickwidth=1, row=2, col=1,
    )
    fig_box.update_yaxes(range=upper_range, tickvals=upper_tickvals,
                          tickfont=dict(size=TICK_FONT_SIZE, family="Arial", color=BLACK),
                          showgrid=False, showline=True, linewidth=3, linecolor=BLACK,
                          ticks="outside", ticklen=6, tickwidth=1, row=1, col=1)
    fig_box.update_yaxes(
        range=lower_range, tickvals=lower_tickvals,
        tickfont=dict(size=TICK_FONT_SIZE, family="Arial", color=BLACK),
        showgrid=False, showline=True, linewidth=3, linecolor=BLACK, ticks="outside",
        ticklen=6, tickwidth=1, row=2, col=1,
    )
    fig_box.add_annotation(x=-0.50, y=lower_range[1], xref="x2", yref="y2", text="//",
                            showarrow=False, font=dict(size=30, color=BLACK, family="Arial"))
    fig_box.add_annotation(x=-0.50, y=upper_range[0], xref="x", yref="y", text="//",
                            showarrow=False, font=dict(size=30, color=BLACK, family="Arial"))

    # A per-subplot y-axis title (row=2 only) would sit centered on the lower subplot alone,
    # not on the combined two-panel axis -- so instead add one standalone, rotated annotation
    # spanning the full figure height (paper coordinates), centered on the whole plot.
    fig_box.add_annotation(
        text=y_axis_title, textangle=-90, x=-0.145, y=0.5, xref="paper", yref="paper",
        xanchor="center", yanchor="middle", showarrow=False,
        font=dict(size=AXIS_TITLE_SIZE, family="Arial", color=BLACK),
    )

    fig_box.show()
    if filename:
        fig_box.write_image(os.path.join(FIGURES_DIR, filename), width=800, height=650, scale=0.8)
        # fig_box.write_image(os.path.join(FIGURES_DIR, filename.rsplit(".", 1)[0] + ".tiff"), width=800, height=650, scale=4.8, format="tiff")  # 600 dpi TIFF (uncomment to export)
    # No return value -- see the note at the end of plot_profile() above.

## 5. Environmental Profiles — Figure 1 & Figure S1

Each variable below produces the same pair of panels: a 24-hour indoor/outdoor mean profile
(Fig. 1a-d, Fig. S1a-b) and a nighttime indoor/outdoor boxplot (Fig. 1e-h, Fig. S1c-d).

Both panels pool at the row level: every reading from every monitored room in both
dormitories contributes equally to the hourly mean/SD and to the boxplot, so a dormitory
with more rooms (or more logged minutes) weighs proportionally more. The profile's shaded
band is therefore a pooled SD across all rooms, and the boxplot whiskers are the full
range (minimum to maximum), not 1.5 x IQR.

### 5.1 Temperature (Figure 1a, 1e)

In [6]:
plot_profile(
    poi="temperature_calibrated", y_axis_title="Mean Temperature (°C)",
    y_range=[27, 37], y_dtick=2, filename="temperature_indoor_outdoor_profile.png",
    label="Temperature",
)
plot_boxplot(
    poi="temperature_calibrated", y_axis_title="Nighttime Temperature (°C)",
    label_decimals=1, filename="temperature_indoor_outdoor_boxplot.png", label="Temperature",
)

[Temperature] Dorm A vs Dorm B significant hours — Indoor: [0, 1, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23]
[Temperature] Dorm A vs Dorm B significant hours — Outdoor: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 22, 23]


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/plotly/io/_kaleido.py:510: DeprecationWarning:


Support for the 'engine' argument is deprecated and will be removed after September 2025.
Kaleido will be the only supported engine at that time.




[Temperature] Nights used: {'dorm_A_indoor': 34, 'dorm_A_outdoor': 34, 'dorm_B_indoor': 42, 'dorm_B_outdoor': 42, 'overall_indoor': 76, 'overall_outdoor': 76}
[Temperature] Indoor nighttime stats: {'n_samples': 142728, 'min': np.float64(27.0350942812983), 'q1': np.float64(30.2050942812983), 'median': np.float64(31.042236162361625), 'q3': np.float64(32.042236162361625), 'max': np.float64(35.59299264705882)}
[Temperature] Outdoor nighttime stats: {'n_samples': 88625, 'min': np.float64(23.5239172072907), 'q1': np.float64(28.328309641532755), 'median': np.float64(29.152991955445543), 'q3': np.float64(30.039022058823527), 'max': np.float64(33.74902205882353)}


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/plotly/io/_kaleido.py:510: DeprecationWarning:


Support for the 'engine' argument is deprecated and will be removed after September 2025.
Kaleido will be the only supported engine at that time.




### 5.2 Relative Humidity (Figure 1c, 1g)

In [7]:
plot_profile(
    poi="humidity_calibrated", y_axis_title="Mean Relative Humidity (%)",
    y_dtick=10, filename="relativehumidity_indoor_outdoor_profile.png", label="Relative Humidity",
)
plot_boxplot(
    poi="humidity_calibrated", y_axis_title="Nighttime Relative Humidity (%)",
    label_decimals=0, filename="relativehumidity_indoor_outdoor_boxplot.png", label="Relative Humidity",
)

[Relative Humidity] Dorm A vs Dorm B significant hours — Indoor: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 22, 23]
[Relative Humidity] Dorm A vs Dorm B significant hours — Outdoor: [0, 1, 2, 3, 4, 5, 6, 7, 8, 13, 14, 16, 17, 18, 19, 20, 22]


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/plotly/io/_kaleido.py:510: DeprecationWarning:


Support for the 'engine' argument is deprecated and will be removed after September 2025.
Kaleido will be the only supported engine at that time.




[Relative Humidity] Nights used: {'dorm_A_indoor': 34, 'dorm_A_outdoor': 34, 'dorm_B_indoor': 42, 'dorm_B_outdoor': 42, 'overall_indoor': 76, 'overall_outdoor': 76}
[Relative Humidity] Indoor nighttime stats: {'n_samples': 142728, 'min': np.float64(44.31814323607427), 'q1': np.float64(70.33038639876352), 'median': np.float64(74.33038639876352), 'q3': np.float64(77.71344667697063), 'max': np.float64(89.33038639876352)}
[Relative Humidity] Outdoor nighttime stats: {'n_samples': 88625, 'min': np.float64(48.78628783199506), 'q1': np.float64(74.04130367624343), 'median': np.float64(79.36242274412855), 'q3': np.float64(83.95795334040297), 'max': np.float64(93.79353632478633)}


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/plotly/io/_kaleido.py:510: DeprecationWarning:


Support for the 'engine' argument is deprecated and will be removed after September 2025.
Kaleido will be the only supported engine at that time.




### 5.3 Absolute Humidity (Figure 1b, 1f)

In [8]:
plot_profile(
    poi="abs_humidity_c", y_axis_title="Mean Absolute Humidity (g/m³)",
    filename="absolutehumidity_indoor_outdoor_profile.png", label="Absolute Humidity",
)
plot_boxplot(
    poi="abs_humidity_c", y_axis_title="Nighttime Absolute Humidity (g/m³)",
    label_decimals=0, filename="absolutehumidity_indoor_outdoor_boxplot.png", label="Absolute Humidity",
)

[Absolute Humidity] Dorm A vs Dorm B significant hours — Indoor: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23]
[Absolute Humidity] Dorm A vs Dorm B significant hours — Outdoor: [0, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23]


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/plotly/io/_kaleido.py:510: DeprecationWarning:


Support for the 'engine' argument is deprecated and will be removed after September 2025.
Kaleido will be the only supported engine at that time.




[Absolute Humidity] Nights used: {'dorm_A_indoor': 34, 'dorm_A_outdoor': 34, 'dorm_B_indoor': 42, 'dorm_B_outdoor': 42, 'overall_indoor': 76, 'overall_outdoor': 76}
[Absolute Humidity] Indoor nighttime stats: {'n_samples': 142728, 'min': np.float64(17.094860842906883), 'q1': np.float64(22.801434160711008), 'median': np.float64(23.816568858727564), 'q3': np.float64(24.735701649562163), 'max': np.float64(27.474849288287174)}
[Absolute Humidity] Outdoor nighttime stats: {'n_samples': 88625, 'min': np.float64(16.693856606850037), 'q1': np.float64(21.96887548418739), 'median': np.float64(22.92229818920185), 'q3': np.float64(23.6231024846803), 'max': np.float64(26.015506407136385)}


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/plotly/io/_kaleido.py:510: DeprecationWarning:


Support for the 'engine' argument is deprecated and will be removed after September 2025.
Kaleido will be the only supported engine at that time.




### 5.4 Wet Bulb Temperature (Figure 1d, 1h)

In [9]:
plot_profile(
    poi="wet_bulb_temp", y_axis_title="Mean Wet Bulb Temperature (°C)",
    y_dtick=1, filename="wetbulb_indoor_outdoor_profile.png", label="Wet Bulb Temperature",
)
plot_boxplot(
    poi="wet_bulb_temp", y_axis_title="Nighttime Wet Bulb Temperature (°C)",
    label_decimals=1, filename="wetbulb_indoor_outdoor_boxplot.png", label="Wet Bulb Temperature",
)

[Wet Bulb Temperature] Dorm A vs Dorm B significant hours — Indoor: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23]
[Wet Bulb Temperature] Dorm A vs Dorm B significant hours — Outdoor: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 23]


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/plotly/io/_kaleido.py:510: DeprecationWarning:


Support for the 'engine' argument is deprecated and will be removed after September 2025.
Kaleido will be the only supported engine at that time.




[Wet Bulb Temperature] Nights used: {'dorm_A_indoor': 34, 'dorm_A_outdoor': 34, 'dorm_B_indoor': 42, 'dorm_B_outdoor': 42, 'overall_indoor': 76, 'overall_outdoor': 76}
[Wet Bulb Temperature] Indoor nighttime stats: {'n_samples': 142728, 'min': np.float64(24.285147037678787), 'q1': np.float64(26.532122664562745), 'median': np.float64(27.21203700610117), 'q3': np.float64(27.865017933804086), 'max': np.float64(29.714351616491395)}
[Wet Bulb Temperature] Outdoor nighttime stats: {'n_samples': 88625, 'min': np.float64(22.662374487804367), 'q1': np.float64(25.633919652247894), 'median': np.float64(26.088128554195695), 'q3': np.float64(26.53928032160294), 'max': np.float64(28.320989764261572)}


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/plotly/io/_kaleido.py:510: DeprecationWarning:


Support for the 'engine' argument is deprecated and will be removed after September 2025.
Kaleido will be the only supported engine at that time.




### 5.5 CO₂ Concentration (Figure S1a, S1c)

In [10]:
plot_profile(
    poi="co2_used", y_axis_title="Mean CO₂ Concentration (ppm)",
    filename="co2_indoor_outdoor_profile.png", label="CO2",
)
plot_boxplot(
    poi="co2_used", y_axis_title="Nighttime CO₂ Concentration (ppm)",
    label_decimals=0, filename="co2_indoor_outdoor_boxplot.png", label="CO2",
)

[CO2] Dorm A vs Dorm B significant hours — Indoor: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23]
[CO2] Dorm A vs Dorm B significant hours — Outdoor: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23]


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/plotly/io/_kaleido.py:510: DeprecationWarning:


Support for the 'engine' argument is deprecated and will be removed after September 2025.
Kaleido will be the only supported engine at that time.




[CO2] Nights used: {'dorm_A_indoor': 34, 'dorm_A_outdoor': 34, 'dorm_B_indoor': 42, 'dorm_B_outdoor': 42, 'overall_indoor': 76, 'overall_outdoor': 76}
[CO2] Indoor nighttime stats: {'n_samples': 142728, 'min': np.float64(395.0), 'q1': np.float64(525.0), 'median': np.float64(598.0), 'q3': np.float64(715.0), 'max': np.float64(1119.0)}
[CO2] Outdoor nighttime stats: {'n_samples': 87983, 'min': np.float64(400.0), 'q1': np.float64(443.0), 'median': np.float64(459.0), 'q3': np.float64(481.0), 'max': np.float64(691.0)}


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/plotly/io/_kaleido.py:510: DeprecationWarning:


Support for the 'engine' argument is deprecated and will be removed after September 2025.
Kaleido will be the only supported engine at that time.




### 5.6 PM₂.₅ Concentration (Figure S1b, S1d)

Outdoor PM₂.₅ includes occasional large spikes (max ≈ 900 µg/m³) that would dwarf the
indoor/outdoor boxes on a linear axis, so the nighttime boxplot uses a broken y-axis. (An
earlier, unbroken-axis version of this same boxplot was dropped as a near-identical
duplicate.)

In [11]:
plot_profile(
    poi="pm25", y_axis_title="Mean PM₂.₅ Concentration (μg/m³)", y_range=[0, 40], y_dtick=10,
    filename="pm25_indoor_outdoor_profile.png", label="PM2.5",
)
plot_boxplot_broken_axis(
    poi="pm25", y_axis_title="Nighttime PM₂.₅ Concentration (μg/m³)",
    lower_range=[0, 30], upper_range=[200, 1000],
    lower_tickvals=[0, 15, 30], upper_tickvals=[200, 1000],
    label_decimals=0, filename="pm25_indoor_outdoor_boxplot.png", label="PM2.5",
)

[PM2.5] Dorm A vs Dorm B significant hours — Indoor: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 18, 19, 20, 21, 22]
[PM2.5] Dorm A vs Dorm B significant hours — Outdoor: [0, 1, 2, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 20, 21]


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/plotly/io/_kaleido.py:510: DeprecationWarning:


Support for the 'engine' argument is deprecated and will be removed after September 2025.
Kaleido will be the only supported engine at that time.




[PM2.5] Nights used: {'dorm_A_indoor': 34, 'dorm_A_outdoor': 34, 'dorm_B_indoor': 42, 'dorm_B_outdoor': 42, 'overall_indoor': 76, 'overall_outdoor': 76}
[PM2.5] Indoor nighttime stats: {'n_samples': 142728, 'min': np.float64(0.6), 'q1': np.float64(8.0), 'median': np.float64(11.6), 'q3': np.float64(17.1), 'max': np.float64(360.9)}
[PM2.5] Outdoor nighttime stats: {'n_samples': 88502, 'min': np.float64(0.9), 'q1': np.float64(9.0), 'median': np.float64(12.7), 'q3': np.float64(18.1), 'max': np.float64(888.3)}


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/plotly/io/_kaleido.py:510: DeprecationWarning:


Support for the 'engine' argument is deprecated and will be removed after September 2025.
Kaleido will be the only supported engine at that time.




## 6. Sleep Data Pipeline

Builds the per-night analytic dataset in four steps, each writing an intermediate CSV to `data/` so the pipeline can be resumed from any stage:
1. **Actigraphy sleep metrics** (Tudor-Locke-scored TIB/TST/SE/SOL/WASO from the Ametris
   watch) → `heats-dorms-sleepperiodmetrics-1h.csv`
2. **Merge subjective (Qualtrics) ratings** (thermal sensation, preference, air movement
   preference) → `...-1h-qualtrics.csv`
3. **Attach the environmental exposure during each sleep period** → `...-1h-qualtrics-env.csv`
4. **Clean & exclude** down to the final analytic sample of 694 nights / 38 participants →
   `...-1h-qualtrics-env-clean.csv`

### 6.1 Actigraphy sleep metrics (file #1)

In [12]:
import pandas as pd
from datetime import timedelta
from pandas.api.types import is_datetime64_any_dtype

# -------------------------------------
# Step 0: Load data
# -------------------------------------
con_sleepdata_filepath = os.path.join(DATA_DIR, "heats-dorms-sleepperiodmetrics.csv")
df = pd.read_csv(con_sleepdata_filepath)
# -------------------------------------
# Step 1: Room assignments (defined once in Section 2, cell "Control-arm
# room -> participant assignments", and reused here)
# -------------------------------------
room_assignments = CON_ROOM_ASSIGNMENTS

# Reverse mapping (subject → room)
subject_to_room = {
    subject: room
    for room, subjects in room_assignments.items()
    for subject in subjects
}

# -------------------------------------
# Step 2: Filter dataset
# -------------------------------------
# Keep only rows with valid logged sleep
if df['SleepDataLogged'].dtype == 'object':
    df_filtered = df[df['SleepDataLogged'].astype(str).str.upper() == 'TRUE']
else:
    df_filtered = df[df['SleepDataLogged'] == True]

# df_filtered = df.copy() # hash this when unhashing above filter

# Keep only participants from the specified rooms
participants = [s for subs in room_assignments.values() for s in subs]
df_filtered = df_filtered[df_filtered['Subject'].isin(participants)]

# Add room column
# df_filtered['Room'] = df_filtered['Subject'].map(subject_to_room)

# -------------------------------------
# Step 3: Select and process relevant columns
# -------------------------------------
cols_to_keep = ['Subject', 'Site', 'Room', 'Adjusted_date', 'Night_WearTime_min', 'SleepDataLogged',
                'InBedTime', 'OutBedTime', 
                'Onset', 'LatencyInMinutes', 'AvgAwakeningInMinutes', 'AwakeningCount', 'Efficiency', 
                'TimeAsleepInMinutes', 'TimeAwakeInMinutes', 'WakeAfterOnsetInMinutes', 'TotalInBedTime_minutes']
df_window = df_filtered[cols_to_keep].copy()

# Multiply Efficiency column by 100
df_window["Efficiency"] = df_window["Efficiency"] * 100

# Convert to datetime. The source CSV stores these as d/m/yy H:MM (e.g.
# "1/5/25 22:21"); an explicit format avoids pandas' per-element dayfirst
# fallback (and its warning) and is faster.
DATETIME_FORMAT = "%d/%m/%y %H:%M"
df_window['InBedTime'] = pd.to_datetime(df_window['InBedTime'], errors='coerce', format=DATETIME_FORMAT)
df_window['OutBedTime'] = pd.to_datetime(df_window['OutBedTime'], errors='coerce', format=DATETIME_FORMAT)
df_window['Onset'] = pd.to_datetime(df_window['Onset'], errors='coerce', format=DATETIME_FORMAT)

# Convert Adjusted_date to yyyy-mm-dd (date only, no time). Source format
# is d/m/yy (e.g. "1/5/25"), same as above but without a time component.
df_window['Adjusted_date'] = pd.to_datetime(
    df_window['Adjusted_date'],
    format="%d/%m/%y",
    errors='coerce'
).dt.date

# Drop rows with invalid timestamps
df_window = df_window.dropna(subset=['InBedTime', 'OutBedTime'])

# -------------------------------------
# Step 4: Create ±1 hour window columns
# -------------------------------------
df_window['InBedTime_1hrbefore'] = df_window['InBedTime'] - timedelta(hours=1)
# df_window['OutBedTime_1hrafter'] = df_window['OutBedTime'] + timedelta(hours=0)

# -------------------------------------
# Step 5: Create day of week column
# -------------------------------------

# Ensure Adjusted_date is datetime
df_window['Adjusted_date'] = pd.to_datetime(
    df_window['Adjusted_date'], errors='coerce'
)

# Create day-of-week column
df_window['Adjusted_day'] = df_window['Adjusted_date'].dt.day_name()

# Move Adjusted_day right after Adjusted_date
cols = df_window.columns.tolist()
idx = cols.index('Adjusted_date')
cols.insert(idx + 1, cols.pop(cols.index('Adjusted_day')))
df_window = df_window.loc[:, cols]

# -------------------------------------
# Step 5: Recalculate and Create TIB_recalculated and SE_recalculated columns
# -------------------------------------

df_window['TIB_recalculated'] = (
    pd.to_numeric(df_window['TimeAsleepInMinutes'], errors='coerce') +
    pd.to_numeric(df_window['WakeAfterOnsetInMinutes'], errors='coerce') +
    pd.to_numeric(df_window['LatencyInMinutes'], errors='coerce'))

df_window['SE_recalculated'] = (
    pd.to_numeric(df_window['TimeAsleepInMinutes'], errors='coerce')  /
    pd.to_numeric(df_window['TIB_recalculated'], errors='coerce')
) * 100

# Move SE_recalculated right after Efficiency
cols = df_window.columns.tolist()
idx_1 = cols.index('Efficiency')
cols.insert(idx_1 + 1, cols.pop(cols.index('SE_recalculated')))
# df_window = df_window.loc[:, cols]

# Move TIB_recalculated right after TotalInBedTime_minutes
idx_2 = cols.index('TotalInBedTime_minutes')
cols.insert(idx_2 + 1, cols.pop(cols.index('TIB_recalculated')))
df_window = df_window.loc[:, cols]

# -------------------------------------
# Step 6: Sort
# -------------------------------------

# Create numeric Subject sort key
df_window['_Subject_num'] = (
    df_window['Subject']
    .str.extract(r'(\d+)')
    .astype(int)
)

# Sort by Subject (numeric), then by Adjusted_date
df_window = (
    df_window
    .sort_values(by=['_Subject_num', 'Adjusted_date'])
    .drop(columns='_Subject_num')
    .reset_index(drop=True)
)

# -------------------------------------
# Step 7: Save to CSV
# -------------------------------------
output_path = os.path.join(OUTPUT_DATA_DIR, "heats-dorms-sleepperiodmetrics-1h.csv")
df_window.to_csv(output_path, index=False)

# -------------------------------------
# Step 8: Summary
# -------------------------------------
print(f"✅ New filtered file saved as:\n{output_path}")
print(f"Total valid rows: {len(df_window)}")
print(f"Unique participants: {df_window['Subject'].nunique()}")
print(f"Rooms included: {df_window['Room'].unique().tolist()}")
print(f"Columns included: {list(df_window.columns)}")


✅ New filtered file saved as:
outputs/data/heats-dorms-sleepperiodmetrics-1h.csv
Total valid rows: 909
Unique participants: 41
Rooms included: ['B206', 'B205', 'B207', 'B102', 'A201', 'B302', 'A104', 'A305']
Columns included: ['Subject', 'Site', 'Room', 'Adjusted_date', 'Adjusted_day', 'Night_WearTime_min', 'SleepDataLogged', 'InBedTime', 'OutBedTime', 'Onset', 'LatencyInMinutes', 'AvgAwakeningInMinutes', 'AwakeningCount', 'Efficiency', 'SE_recalculated', 'TimeAsleepInMinutes', 'TimeAwakeInMinutes', 'WakeAfterOnsetInMinutes', 'TotalInBedTime_minutes', 'TIB_recalculated', 'InBedTime_1hrbefore']


### 6.2 Merge subjective (Qualtrics) survey ratings (file #2)

In [13]:
import pandas as pd

# --------------------
# File paths
# --------------------
con_sleepdata_window_filepath = os.path.join(OUTPUT_DATA_DIR, "heats-dorms-sleepperiodmetrics-1h.csv")
dorm_A_qualtrics_filepath = os.path.join(DATA_DIR, "dorm_A_qualtrics.csv")
dorm_B_qualtrics_filepath = os.path.join(DATA_DIR, "dorm_B_qualtrics.csv")

output_w_qualtrics_path = os.path.join(OUTPUT_DATA_DIR, "heats-dorms-sleepperiodmetrics-1h-qualtrics.csv")

# --------------------
# Load data
# --------------------
sleep_df = pd.read_csv(con_sleepdata_window_filepath)
dorm_A_q = pd.read_csv(dorm_A_qualtrics_filepath)
dorm_B_q = pd.read_csv(dorm_B_qualtrics_filepath)

# --------------------
# Normalize join keys
# --------------------
for df in [sleep_df, dorm_A_q, dorm_B_q]:
    df['Subject'] = df['Subject'].astype(str).str.strip().str.upper()
    df['Adjusted_date'] = pd.to_datetime(
        df['Adjusted_date'], errors='coerce'
    ).dt.date

# --------------------
# Sanity checks
# --------------------
print("\n--- SANITY CHECKS ---")
print("Sleep rows:", len(sleep_df))
print("Dorm A rows:", len(dorm_A_q))
print("Dorm B rows:", len(dorm_B_q))

sleep_keys = set(zip(sleep_df['Subject'], sleep_df['Adjusted_date']))
dorm_A_keys = set(zip(dorm_A_q['Subject'], dorm_A_q['Adjusted_date']))
dorm_B_keys = set(zip(dorm_B_q['Subject'], dorm_B_q['Adjusted_date']))

print("Sleep \u2229 Dorm A matches:", len(sleep_keys & dorm_A_keys))
print("Sleep \u2229 Dorm B matches:", len(sleep_keys & dorm_B_keys))

# --------------------
# Qualtrics columns to attach to sleep_df
# --------------------
qualtrics_cols = [
    "night_survey_timestamp",
    "morning_survey_timestamp",
    "Q0.2 BEF / AFT",
    "Q1.3 PRE-RTS",
    "Q1.4 RTP",
    "Q1.5 Wind",
]
missing_A = [c for c in qualtrics_cols if c not in dorm_A_q.columns]
missing_B = [c for c in qualtrics_cols if c not in dorm_B_q.columns]
if missing_A or missing_B:
    raise KeyError(
        f"Expected Qualtrics columns missing -- Dorm A: {missing_A}, Dorm B: {missing_B}"
    )

# --------------------
# Merge -- dorm_A_qualtrics.csv and dorm_B_qualtrics.csv
# --------------------
qualtrics_all = pd.concat(
    [dorm_A_q[["Subject", "Adjusted_date"] + qualtrics_cols],
     dorm_B_q[["Subject", "Adjusted_date"] + qualtrics_cols]],
    ignore_index=True,
)

dup_keys = qualtrics_all.duplicated(subset=["Subject", "Adjusted_date"]).sum()
assert dup_keys == 0, (
    f"\u274c {dup_keys} duplicate (Subject, Adjusted_date) keys across the two "
    f"Qualtrics files -- Dorm A/B Subject ranges are expected to be disjoint."
)

sleep_df = sleep_df.merge(
    qualtrics_all,
    on=["Subject", "Adjusted_date"],
    how="left",
)

sleep_df["QualtricsLogged"] = sleep_df[qualtrics_cols].notna().any(axis=1)

print("\n--- FINAL CHECK ---")
print("QualtricsLogged TRUE count:", sleep_df["QualtricsLogged"].sum())

# Uppercase Site (e.g. "Site_A" -> "SITE_A") -- the environmental
# exposure-attachment step (Section 7.3) matches on this exact uppercased
# form.
sleep_df["Site"] = sleep_df["Site"].astype(str).str.upper()

# --------------------
# Save
# --------------------
sleep_df.to_csv(output_w_qualtrics_path, index=False)

print("\n\u2705 Qualtrics data successfully merged and saved.")


--- SANITY CHECKS ---
Sleep rows: 909
Dorm A rows: 304
Dorm B rows: 539
Sleep ∩ Dorm A matches: 301
Sleep ∩ Dorm B matches: 536

--- FINAL CHECK ---
QualtricsLogged TRUE count: 847

✅ Qualtrics data successfully merged and saved.


### 6.3 Attach environmental exposure during each sleep period (file #3)

For each sleep period, indoor (and, for temperature, outdoor) environmental statistics are
computed over the window `InBedTime − 1 hour` through `OutBedTime`, for temperature,
relative humidity, absolute humidity, wet bulb temperature, CO₂,
PM₁, PM₂.₅ and PM₁₀.

In [14]:
import pandas as pd
import numpy as np
from pandas.api.types import is_datetime64_any_dtype
from datetime import datetime

# -------------------------------------
# Step 1: File paths
# -------------------------------------
sleep_file_path = os.path.join(OUTPUT_DATA_DIR, "heats-dorms-sleepperiodmetrics-1h-qualtrics.csv")
output_path = os.path.join(OUTPUT_DATA_DIR, "heats-dorms-sleepperiodmetrics-1h-qualtrics-env.csv")

# -------------------------------------
# Step 2: Load sleep data
# -------------------------------------
sleep_df = pd.read_csv(sleep_file_path)

# -------------------------------------
# Step 3: Ensure proper datetime formats (sleep)
# -------------------------------------
# Convert to datetime in default yyyy-mm-dd HH:MM format
sleep_df['InBedTime_1hrbefore'] = pd.to_datetime(
    sleep_df['InBedTime_1hrbefore'], errors='coerce'
)
sleep_df['OutBedTime'] = pd.to_datetime(
    sleep_df['OutBedTime'], errors='coerce'
)

# -------------------------------------
# Step 4: Prepare environmental dataframes
# -------------------------------------
def prepare_env_df(df, site):
    df = df.copy()
    
    # Ensure index is datetime
    if not isinstance(df.index, pd.DatetimeIndex):
        df.index = pd.to_datetime(df.index, errors='coerce')
    if df.index.tz is not None:
        df.index = df.index.tz_localize(None)
    
    # Ensure id_room exists
    if 'id_room' not in df.columns:
        raise ValueError(f"❌ 'id_room' missing in {site} dataframe")
    df['id_room'] = df['id_room'].astype(str)
    
    # Sort by datetime index
    df = df.sort_index()
    
    return df

# Example: assume dorm_A_condata_df and dorm_B_condata_df already loaded
dorm_A_env_df = prepare_env_df(dorm_A_condata_df0, site="SITE A")
dorm_B_env_df = prepare_env_df(dorm_B_condata_df0, site="SITE B")

print(dorm_A_condata_df0.columns)
print(dorm_B_condata_df0.columns)

# # Ensure dorm A has co2_corrected even if it wasn't produced by prepare_env_df
# if "co2_corrected" not in dorm_A_env_df.columns and "co2" in dorm_A_env_df.columns:
#     dorm_A_env_df["co2_corrected"] = dorm_A_env_df["co2"]

# -------------------------------------
# Step 5: Environmental variables
# -------------------------------------
env_vars = ['temperature_calibrated', 'humidity_calibrated', 'abs_humidity_c', 'wet_bulb_temp', 'co2_used', 'pm1', 'pm25', 'pm10']

# Keep only variables present in BOTH datasets
env_vars = [v for v in env_vars if v in dorm_A_env_df.columns and v in dorm_B_env_df.columns]
if not env_vars:
    raise ValueError("❌ No common environmental variables found")

# Temperature is the only variable that also gets Outdoor and Indoor-Outdoor stats.
# Site A outdoor readings live under a fixed id_room ("Outside 1"); Site B outdoor
# readings live under "{Room} Outdoor" (mirroring the "{Room} Indoor" id used below).
TEMP_VAR = 'temperature_calibrated'
DORM_A_OUTDOOR_ROOM = "Outside 1"
STAT_SUFFIXES = ['mean', 'std', 'median', 'min', 'max', 'IQR_0.75', 'IQR_0.25']

# -------------------------------------
# Step 6: Helper to compute stats
# -------------------------------------
def compute_env_stats(subset, variables):
    stats = {}
    for var in variables:
        series = subset[var].dropna()
        if series.empty:
            continue
        stats[f'{var}_mean'] = series.mean()
        stats[f'{var}_std'] = series.std()
        stats[f'{var}_median'] = series.median()
        stats[f'{var}_min'] = series.min()
        stats[f'{var}_max'] = series.max()
        stats[f'{var}_IQR_0.75'] = series.quantile(0.75)
        stats[f'{var}_IQR_0.25'] = series.quantile(0.25)
    return pd.Series(stats)

# -------------------------------------
# Step 7: Iterate per sleep period (SITE-AWARE)
# -------------------------------------
results = []

for _, row in sleep_df.iterrows():

    site = row.get('Site')
    room_raw = row.get('Room')

    start = row.get('InBedTime_1hrbefore')
    end = row.get('OutBedTime')

    if pd.isna(site) or pd.isna(room_raw) or pd.isna(start) or pd.isna(end):
        results.append(pd.Series({}))
        continue

    # Select correct environmental dataframe and indoor/outdoor room ids
    if site == "SITE_A":
        env_df = dorm_A_env_df
        indoor_room = room_raw
        outdoor_room = DORM_A_OUTDOOR_ROOM
    elif site == "SITE_B":
        env_df = dorm_B_env_df
        indoor_room = f"{room_raw} Indoor"
        outdoor_room = f"{room_raw} Outdoor"
    else:
        results.append(pd.Series({}))
        continue

    # Subset indoor environment data for that room and period
    indoor_subset = env_df.loc[
        (env_df['id_room'] == indoor_room) &
        (env_df.index >= start) &
        (env_df.index <= end)
    ]

    if indoor_subset.empty:
        results.append(pd.Series({}))
        continue

    stats = compute_env_stats(indoor_subset, env_vars)

    # Outdoor temperature + Indoor-Outdoor difference (temperature only)
    outdoor_subset = env_df.loc[
        (env_df['id_room'] == outdoor_room) &
        (env_df.index >= start) &
        (env_df.index <= end)
    ]

    if not outdoor_subset.empty and TEMP_VAR in outdoor_subset.columns:
        outdoor_stats = compute_env_stats(outdoor_subset, [TEMP_VAR])
        outdoor_stats = outdoor_stats.rename(lambda k: f"{k}_Outdoor")
        stats = pd.concat([stats, outdoor_stats])

        for suffix in STAT_SUFFIXES:
            indoor_key = f"{TEMP_VAR}_{suffix}"
            outdoor_key = f"{TEMP_VAR}_{suffix}_Outdoor"
            if indoor_key in stats and outdoor_key in stats:
                stats[f"{TEMP_VAR}_{suffix}_IndoorOutdoorDiff"] = stats[indoor_key] - stats[outdoor_key]

    results.append(stats)

# -------------------------------------
# Step 8: Merge results back
# -------------------------------------
stats_df = pd.DataFrame(results)
merged_df = pd.concat(
    [sleep_df.reset_index(drop=True), stats_df.reset_index(drop=True)],
    axis=1
)

# -------------------------------------
# Step 9: Ensure all datetime columns in yyyy-mm-dd format
# -------------------------------------
datetime_cols = merged_df.select_dtypes(include=['datetime64[ns]', 'datetime64']).columns
for col in datetime_cols:
    merged_df[col] = merged_df[col].dt.strftime('%Y-%m-%d %H:%M')

# -------------------------------------
# Step 10: Save output
# -------------------------------------
merged_df.to_csv(output_path, index=False)

# -------------------------------------
# Step 11: Summary
# -------------------------------------
today_date = datetime.today().strftime('%Y-%m-%d')
current_time = datetime.now().strftime('%H:%M:%S')

print(f"✅ Code has run as of {today_date}, {current_time}")
print(f"✅ Environmental stats added for variables: {env_vars}")
print(f"✅ Outdoor + Indoor-Outdoor diff stats added for: {TEMP_VAR}")
print(f"✅ Rows processed: {len(merged_df)}")
print(f"✅ Output saved to:\n{output_path}")

Index(['abs_humidity', 'ch2o', 'co2', 'device_name', 'eiaqi', 'humidity',
       'iaqi', 'light', 'noise', 'noxindex', 'pm1', 'pm10', 'pm25', 'pm4',
       'pressure', 'product_id', 'tci', 'temperature', 'voc', 'vocindex',
       'temperature_calibrated', 'humidity_calibrated', 'id_room',
       'wet_bulb_temp', 'abs_humidity_c', 'co2_used'],
      dtype='object')
Index(['abs_humidity', 'ch2o', 'co2', 'device_name', 'eiaqi', 'humidity',
       'iaqi', 'light', 'noise', 'noxindex', 'pm1', 'pm10', 'pm25', 'pm4',
       'pressure', 'product_id', 'tci', 'temperature', 'voc', 'vocindex',
       'temperature_calibrated', 'humidity_calibrated', 'co2_corrected',
       'id_room', 'wet_bulb_temp', 'abs_humidity_c', 'co2_used'],
      dtype='object')
✅ Code has run as of 2026-09-18, 09:19:02
✅ Environmental stats added for variables: ['temperature_calibrated', 'humidity_calibrated', 'abs_humidity_c', 'wet_bulb_temp', 'co2_used', 'pm1', 'pm25', 'pm10']
✅ Outdoor + Indoor-Outdoor diff stats added 

### 6.4 Data cleaning & exclusion cascade → final analytic sample

In [15]:
#updated code
import pandas as pd
import numpy as np
from scipy import stats
import statsmodels.formula.api as smf
from pandas.api.types import is_numeric_dtype

# ---------------------------
# File paths
# ---------------------------
previous_file = os.path.join(OUTPUT_DATA_DIR, "heats-dorms-sleepperiodmetrics-1h-qualtrics-env.csv")
manuallyremove_file = os.path.join(DATA_DIR, "heats-dorms-manualreviewnights.xlsx")

output_file = os.path.join(OUTPUT_DATA_DIR, "heats-dorms-sleepperiodmetrics-1h-qualtrics-env-clean.csv")
removed_rows_output = os.path.join(OUTPUT_DATA_DIR, "heats-dorms-condata-rows_removed.csv")

# ---------------------------
# Helper function to log removed rows
# ---------------------------
removed_rows_log = []

def log_removed_rows(df_removed_subset, reason):
    """
    Append removed rows to the global removal log.
    Keeps only Subject, Adjusted_date, Removed_by.
    """
    global removed_rows_log

    if df_removed_subset is None or df_removed_subset.empty:
        return

    cols_needed = ["Subject", "Adjusted_date"]
    missing_cols = [c for c in cols_needed if c not in df_removed_subset.columns]
    if missing_cols:
        raise KeyError(f"Missing required columns for removal log: {missing_cols}")

    temp = df_removed_subset[["Subject", "Adjusted_date"]].copy()
    temp["Adjusted_date"] = pd.to_datetime(temp["Adjusted_date"], errors="coerce").dt.normalize()
    temp["Removed_by"] = reason

    removed_rows_log.append(temp)

# ---------------------------
# Load data
# ---------------------------
df = pd.read_csv(
    previous_file,
    parse_dates=['InBedTime_1hrbefore', 'OutBedTime'],
)

# Standardize Adjusted_date early
df["Adjusted_date"] = pd.to_datetime(df["Adjusted_date"], errors="coerce").dt.normalize()

# ---------------------------
# Define expected columns
# ---------------------------
env_vars = [
    'temperature_calibrated_mean','temperature_calibrated_std','temperature_calibrated_median','temperature_calibrated_min','temperature_calibrated_max',
    'temperature_calibrated_IQR_0.75','temperature_calibrated_IQR_0.25',
    'humidity_calibrated_mean','humidity_calibrated_std','humidity_calibrated_median','humidity_calibrated_min','humidity_calibrated_max','humidity_calibrated_IQR_0.75','humidity_calibrated_IQR_0.25',
    'abs_humidity_c_mean', 'abs_humidity_c_std','abs_humidity_c_median','abs_humidity_c_min','abs_humidity_c_max','abs_humidity_c_IQR_0.75','abs_humidity_c_IQR_0.25',
    'wet_bulb_temp_mean','wet_bulb_temp_std','wet_bulb_temp_median','wet_bulb_temp_min','wet_bulb_temp_max','wet_bulb_temp_IQR_0.75','wet_bulb_temp_IQR_0.25',
    'co2_used_mean','co2_used_std','co2_used_median','co2_used_min','co2_used_max','co2_used_IQR_0.75','co2_used_IQR_0.25',
    'pm25_mean','pm25_std','pm25_median','pm25_min','pm25_max','pm25_IQR_0.75','pm25_IQR_0.25',
]
# ---------------------------
# [1] Remove duplicate rows based on Subject + Adjusted_date
# ---------------------------
# Count rows before removal
rows_before = len(df)

dup_mask = df.duplicated(subset=["Subject", "Adjusted_date"], keep="first")

# Log rows that will be removed
df_removed_duplicates = df.loc[dup_mask, ['Subject', 'Adjusted_date']].copy()
log_removed_rows(df_removed_duplicates, "duplicated Subject–Adjusted_date rows")

# Remove duplicates
df = df.loc[~dup_mask].copy()

print(f"✅ TOTAL ROWS BEFORE: {rows_before}")
print(f"🚫 {dup_mask.sum()} duplicated rows removed")

# ---------------------------
# [1] Remove rows with 27 Sept's data
# Dorm B's sensors only start logging at 02:00 on 28 Sep (the study's first
# night there), so any night anchored to 27 Sep has no pre-sleep (InBedTime-1h
# onward) environmental exposure at all and cannot be analyzed.
# ---------------------------
before = len(df)

mask_remove_27sept = df['Adjusted_date'] == pd.Timestamp('2025-09-27')
df_removed_27sept = df.loc[mask_remove_27sept, ['Subject', 'Adjusted_date']].copy()
log_removed_rows(df_removed_27sept, "Adjusted_date = 2025-09-27")

df = df.loc[~mask_remove_27sept].copy()

after = len(df)
print(f"\n🚫 {before-after} rows removed with Adjusted_date = 2025-09-27")
print(f"✅ Remaining valid nights for analysis: {len(df)}")


# ---------------------------
# [2] Filter out rows missing environmental data
# ---------------------------
sentinel = 'temperature_calibrated_mean'
excluded_mask = df[sentinel].isna()
excluded_rows = df.loc[excluded_mask, ['Subject', 'Adjusted_date']].copy()

# print("\nRows with missing environmental data:")
# if len(excluded_rows) > 0:
#     display_cols = ['Subject', 'Adjusted_date']
#     extra_display_col = 'Room' if 'Room' in df.columns else None
#     if extra_display_col:
#         print(df.loc[excluded_mask, ['Subject', 'Room', 'Adjusted_date']].to_string(index=False))
#     else:
#         print(excluded_rows.to_string(index=False))
# else:
#     print("None found — all rows contain environmental data.")

log_removed_rows(excluded_rows, "missing environmental data")

df_valid = df.loc[~excluded_mask].copy()
print(f"\n🚫 {len(excluded_rows)} rows removed due to missing environmental data")
print(f"✅ Remaining valid nights for analysis: {len(df_valid)}")

# ---------------------------
# Row count + QualtricsLogged summary
# ---------------------------
total_rows = len(df_valid)

qualtrics_true_mask = df_valid['QualtricsLogged'].astype(str).str.upper() == 'TRUE'
true_rows = qualtrics_true_mask.sum()
false_rows = (~qualtrics_true_mask).sum()

df_removed_qualtrics = df_valid.loc[~qualtrics_true_mask, ['Subject', 'Adjusted_date']].copy()
log_removed_rows(df_removed_qualtrics, "QualtricsLogged != TRUE")

print(f"\n🚫 {total_rows - true_rows} rows removed due to QualtricsLogged != TRUE")

# ---------------------------
# [3] Remove rows if there are no Qualtrics data
# ---------------------------
df_valid = df_valid.loc[qualtrics_true_mask].copy()
print(f"✅ Remaining valid nights for analysis: {len(df_valid)}")

# -----------------------------
# Columns to check
#
# "Missing survey data" is defined as: no survey response at all
# (QualtricsLogged == False, already removed above) OR a blank value in any
# of the columns below -- the sleep-metrics fields actually used downstream,
# plus the retained Qualtrics response columns.
# -----------------------------
cols_to_check = [
    "InBedTime", "OutBedTime", "Onset",
    "LatencyInMinutes", "AvgAwakeningInMinutes", "AwakeningCount",
    "Efficiency", "SE_recalculated", "TimeAsleepInMinutes",
    "TimeAwakeInMinutes", "WakeAfterOnsetInMinutes",
    "TotalInBedTime_minutes", "TIB_recalculated",
    "InBedTime_1hrbefore", "QualtricsLogged",
    "night_survey_timestamp", "morning_survey_timestamp",
    "Q0.2 BEF / AFT", "Q1.3 PRE-RTS", "Q1.4 RTP", "Q1.5 Wind"
]

missing_check_cols = [col for col in cols_to_check if col not in df_valid.columns]
if missing_check_cols:
    raise KeyError(
        f"\u274c cols_to_check references columns not present in df_valid: "
        f"{missing_check_cols}. Update cols_to_check (or the upstream pipeline) "
        f"rather than silently skipping them."
    )
cols_to_check_existing = cols_to_check

# -----------------------------
# Normalize blanks (NaN, empty, whitespace)
# -----------------------------
df_check = df_valid.copy()

for col in cols_to_check_existing:
    df_check[col] = df_check[col].replace(r"^\s*$", np.nan, regex=True)

# -----------------------------
# Summary counts
# -----------------------------
total_rows = len(df_check)
rows_with_any_blank = df_check[cols_to_check_existing].isna().any(axis=1)
n_rows_with_blanks = rows_with_any_blank.sum()

# Keep only complete rows
df_removed_incomplete = df_valid.loc[rows_with_any_blank, ['Subject', 'Adjusted_date']].copy()
log_removed_rows(df_removed_incomplete, "incomplete Qualtrics data")

df_valid = df_valid.loc[~rows_with_any_blank].copy()

print(f"\n🚫 {rows_with_any_blank.sum()} rows removed that have incomplete Qualtrics data")
print(f"✅ Remaining rows: {len(df_valid)}")

# ---------------------------
# [4] Remove nights which should be manually removed
# ---------------------------
df_remove = pd.read_excel(manuallyremove_file, usecols=["Subject", "Adjusted_date"])

# Both sides are already ISO-format dates by this point (df_valid's came
# through to_csv()/read_csv() upstream; df_remove's are native Excel dates
# read via read_excel()), so no dayfirst/format hint is needed here.
# infer_datetime_format is dropped: it's deprecated and a no-op on pandas'
# current parser, which already infers a per-column format automatically.
df_valid["Adjusted_date"] = pd.to_datetime(
    df_valid["Adjusted_date"], errors="coerce"
).dt.normalize()

df_remove["Adjusted_date"] = pd.to_datetime(
    df_remove["Adjusted_date"], errors="coerce"
).dt.normalize()

assert pd.api.types.is_datetime64_any_dtype(df_valid["Adjusted_date"]), \
    "❌ df_valid Adjusted_date is NOT datetime"

assert pd.api.types.is_datetime64_any_dtype(df_remove["Adjusted_date"]), \
    "❌ df_remove Adjusted_date is NOT datetime"

remove_keys = set(zip(df_remove["Subject"], df_remove["Adjusted_date"]))

df_valid["_to_remove"] = list(zip(df_valid["Subject"], df_valid["Adjusted_date"]))
df_valid["_to_remove"] = df_valid["_to_remove"].isin(remove_keys)

df_removed_manual = df_valid[df_valid["_to_remove"]].drop(columns="_to_remove")
df_ready = df_valid[~df_valid["_to_remove"]].drop(columns="_to_remove")

# Log rows removed from manuallyremove_file
log_removed_rows(df_removed_manual, "manually removed after checking Actograms")

check = df_ready.merge(
    df_remove,
    on=["Subject", "Adjusted_date"],
    how="inner"
)

removed_counts_manual = (
    df_removed_manual
    .groupby("Subject")
    .size()
    .sort_values(ascending=False)
)

total_nights_manual = removed_counts_manual.sum()
total_subjects_manual = removed_counts_manual.shape[0]

print(f"\n🚫 {total_nights_manual} rows removed across {total_subjects_manual} subjects after manually checking Actograms")
print(f"✅ Remaining valid nights for analysis after manual removal: {len(df_ready)}")

# ============================================================
# [5] ±3 SD filtering for extreme sleep values -- single pass
#
# Mean/SD for each of the 5 metrics is computed once, on the same
# pre-outlier sample (df_ready). A night is excluded if it is beyond
# ±3 SD on ANY of the 5 metrics, and all flagged nights are removed.
# ============================================================

print("\n" + "=" * 70)
print("EXTREME VALUE CLEANING (±3 SD, single pass)")
print("=" * 70)

filter_metrics = [
    ("TimeAsleepInMinutes", "TST"),
    ("TIB_recalculated", "TIB"),
    ("SE_recalculated", "SE"),
    ("LatencyInMinutes", "SOL"),
    ("WakeAfterOnsetInMinutes", "WASO"),
]

df_clean = df_ready.copy()
df_clean["Adjusted_date"] = pd.to_datetime(df_clean["Adjusted_date"], errors="coerce").dt.normalize()

initial_nights = len(df_clean)

required_cols = ["Subject", "Adjusted_date"]
missing_required = [c for c in required_cols if c not in df_clean.columns]
if missing_required:
    raise KeyError(f"Missing required columns for removed-rows output: {missing_required}")

missing_metric_cols = [c for c, _ in filter_metrics if c not in df_clean.columns]
if missing_metric_cols:
    raise KeyError(f"Missing sleep-metric columns for ±3 SD filtering: {missing_metric_cols}")

outlier_mask = pd.Series(False, index=df_clean.index)
removed_counts = {}

for col, label in filter_metrics:
    mean_val = df_clean[col].mean()
    sd_val = df_clean[col].std()
    lower = mean_val - 3 * sd_val
    upper = mean_val + 3 * sd_val

    is_outlier = df_clean[col].isna() | (df_clean[col] < lower) | (df_clean[col] > upper)
    removed_counts[label] = int(is_outlier.sum())
    outlier_mask |= is_outlier

    print(f"{label:5s} beyond ±3 SD (mean={mean_val:.2f}, sd={sd_val:.2f}): {removed_counts[label]} nights")

removed_this_step = df_clean.loc[outlier_mask, ["Subject", "Adjusted_date"]].copy()
log_removed_rows(removed_this_step, "beyond +- 3 SD (single pass, any of TST/TIB/SE/SOL/WASO)")

df_clean = df_clean.loc[~outlier_mask].copy()

after_n = len(df_clean)
print(f"\n🚫 {int(outlier_mask.sum())} nights removed beyond ±3 SD on any of the 5 metrics (single pass)")
print(f"✅ Remaining nights {after_n}")

# ------------------------------------------------------------
# [6] Remove 2 subjects' data
# CDS025 and CDS038
# ------------------------------------------------------------
subjects_to_remove = ['CDS025', 'CDS038']

rows_before = len(df_clean)

mask_remove_subjects = df_clean['Subject'].isin(subjects_to_remove)
df_removed_subjects = df_clean.loc[mask_remove_subjects, ['Subject', 'Adjusted_date']].copy()
log_removed_rows(df_removed_subjects, "eliminating CDS025 and CDS038")

df_clean = df_clean.loc[~mask_remove_subjects].copy()

rows_after = len(df_clean)
rows_removed = rows_before - rows_after

print(f"\n🚫 {rows_removed} rows removed after eliminating CDS025 and CDS038")
print(f"✅ FINAL rows remaining: {rows_after}")

# ------------------------------------------------------------
# Final summary
# ------------------------------------------------------------
final_nights = len(df_clean)
total_removed = initial_nights - final_nights

# ------------------------------------------------------------
# Save all removed rows to CSV
# ------------------------------------------------------------
if removed_rows_log:
    df_removed_all = pd.concat(removed_rows_log, ignore_index=True)

    df_removed_all["Adjusted_date"] = pd.to_datetime(
        df_removed_all["Adjusted_date"], errors="coerce"
    ).dt.normalize()

    df_removed_all = df_removed_all.sort_values(
        by=["Subject", "Adjusted_date", "Removed_by"],
        na_position="last"
    ).reset_index(drop=True)

    df_removed_all.to_csv(removed_rows_output, index=False)

    print("\n" + "=" * 70)
    print("ALL REMOVED ROWS SAVED")
    print("=" * 70)
    print(f"Saved to: {removed_rows_output}")
    # print("\nPreview:")
    # print(df_removed_all.head(20))
else:
    df_removed_all = pd.DataFrame(columns=["Subject", "Adjusted_date", "Removed_by"])
    df_removed_all.to_csv(removed_rows_output, index=False)

    print("\nNo rows were removed. Empty file saved:")
    print(removed_rows_output)

# ------------------------------------------------------------
# Save final cleaned dataset
# ------------------------------------------------------------
df_clean.to_csv(output_file, index=False)
print(f"\ndf_clean saved to:\n{output_file}")


✅ TOTAL ROWS BEFORE: 909
🚫 10 duplicated rows removed

🚫 9 rows removed with Adjusted_date = 2025-09-27
✅ Remaining valid nights for analysis: 890

🚫 4 rows removed due to missing environmental data
✅ Remaining valid nights for analysis: 886

🚫 61 rows removed due to QualtricsLogged != TRUE
✅ Remaining valid nights for analysis: 825

🚫 2 rows removed that have incomplete Qualtrics data
✅ Remaining rows: 823

🚫 93 rows removed across 15 subjects after manually checking Actograms
✅ Remaining valid nights for analysis after manual removal: 730

EXTREME VALUE CLEANING (±3 SD, single pass)
TST   beyond ±3 SD (mean=335.71, sd=59.68): 4 nights
TIB   beyond ±3 SD (mean=445.99, sd=66.21): 6 nights
SE    beyond ±3 SD (mean=75.61, sd=10.26): 3 nights
SOL   beyond ±3 SD (mean=13.63, sd=17.42): 21 nights
WASO  beyond ±3 SD (mean=96.65, sd=48.62): 6 nights

🚫 33 nights removed beyond ±3 SD on any of the 5 metrics (single pass)
✅ Remaining nights 697

🚫 3 rows removed after eliminating CDS025 and CDS

## 7. Assumption Checks

### 7.1 Normality Checks (Shapiro–Wilk)

Run on the final analytic sample (`df_clean`, 694 nights / 38 participants).

In [16]:
from scipy.stats import shapiro

REQ = [
    "Subject", "Room", "TimeAsleepInMinutes",
    "temperature_calibrated_mean", "humidity_calibrated_mean", "abs_humidity_c_mean",
    "wet_bulb_temp_mean", "co2_used_mean", "pm25_mean", "pm1_mean", "pm10_mean",
    "temperature_calibrated_median", "humidity_calibrated_median", "abs_humidity_c_median",
    "wet_bulb_temp_median", "co2_used_median", "pm25_median", "pm1_median", "pm10_median",
]
missing = [c for c in REQ if c not in df_clean.columns]
assert not missing, f"Missing columns: {missing}"

dc = df_clean

numeric_cols = [
    "TimeAsleepInMinutes", "LatencyInMinutes", "Efficiency", "SE_recalculated", "WakeAfterOnsetInMinutes",
    "TotalInBedTime_minutes", "TIB_recalculated",
    "temperature_calibrated_mean", "humidity_calibrated_mean", "abs_humidity_c_mean",
    "wet_bulb_temp_mean", "co2_used_mean", "pm25_mean", "pm1_mean", "pm10_mean",
    "temperature_calibrated_median", "humidity_calibrated_median", "abs_humidity_c_median",
    "wet_bulb_temp_median", "co2_used_median", "pm25_median", "pm1_median", "pm10_median",
]

print("\n— NORMALITY CHECK (Shapiro–Wilk) —")
for col in numeric_cols:
    if col in dc.columns:
        data = dc[col].dropna()
        if len(data) < 3:
            print(f"{col}: Not enough data for normality test")
            continue
        stat, p = shapiro(data)
        normality = "≈ normal" if p > 0.05 else "non-normal"
        print(f"{col}: W={stat:.3f}, p={p:.3f} → {normality}")


— NORMALITY CHECK (Shapiro–Wilk) —
TimeAsleepInMinutes: W=0.997, p=0.172 → ≈ normal
LatencyInMinutes: W=0.792, p=0.000 → non-normal
Efficiency: W=0.992, p=0.001 → non-normal
SE_recalculated: W=0.992, p=0.001 → non-normal
WakeAfterOnsetInMinutes: W=0.986, p=0.000 → non-normal
TotalInBedTime_minutes: W=0.996, p=0.051 → ≈ normal
TIB_recalculated: W=0.994, p=0.007 → non-normal
temperature_calibrated_mean: W=0.981, p=0.000 → non-normal
humidity_calibrated_mean: W=0.972, p=0.000 → non-normal
abs_humidity_c_mean: W=0.988, p=0.000 → non-normal
wet_bulb_temp_mean: W=0.988, p=0.000 → non-normal
co2_used_mean: W=0.978, p=0.000 → non-normal
pm25_mean: W=0.917, p=0.000 → non-normal
pm1_mean: W=0.917, p=0.000 → non-normal
pm10_mean: W=0.917, p=0.000 → non-normal
temperature_calibrated_median: W=0.982, p=0.000 → non-normal
humidity_calibrated_median: W=0.971, p=0.000 → non-normal
abs_humidity_c_median: W=0.990, p=0.000 → non-normal
wet_bulb_temp_median: W=0.991, p=0.000 → non-normal
co2_used_median:

### 7.2 Collinearity Among Environmental Exposures

Wet bulb temperature is a deterministic function of temperature and relative humidity, and
absolute humidity is closely related to both — so temperature, absolute humidity and WBT are
close to collinear by construction. This cell quantifies that with pairwise Pearson
correlations (pooled and within-person), variance inflation factors, and R² of WBT on
temperature + RH, to support whatever collinearity statement the Discussion reports.

In [17]:
# === Pairwise collinearity among thermal-exposure metrics ===
# Variables: temperature_calibrated_median, humidity_calibrated_median,
#            abs_humidity_c_median, wet_bulb_temperature_median
#
# Purpose: WBT is a deterministic function of T and RH, so T, AH and WBT are
# close to collinear by construction. This script produces a supplementary
# table + figure that lets a reader see these four columns as different views
# of ONE underlying exposure, rather than four separable predictors.
#
# It reports FOUR complementary pieces of evidence, because a bare Pearson
# correlation matrix is descriptive but not, on its own, a full collinearity
# diagnostic on its own). The first
# two go into Panel A of the supplementary table, the third into Panel B —
# both panels ship together in one CSV / printed table:
#   1) Pooled (raw) pairwise Pearson r  -> the standard "correlation table"
#   2) Within-person pairwise Pearson r -> matches the person-demeaned
#      exposure actually used in the OLS models (df0["<var>_within"])
#   3) Variance Inflation Factors (VIF), pooled and within-person -> the
#      formal multicollinearity diagnostic; a large VIF is the quantitative
#      version of "these are not separable effects"
#   4) R^2 from regressing WBT jointly on T + RH -> quantifies "deterministic
#      function of" numerically (should be ~1.0 if WBT was computed from a
#      formula on the same T/RH fields)
#
# Setup: pip install plotly kaleido
# Static PNG export (write_image) needs a Chrome/Chromium binary. If on
# kaleido>=1 and get a "Kaleido requires Chrome" error, run `plotly_get_chrome`
# once (downloads a copy), or point it at an existing Chrome/Chromium install
# already on machine.

import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.outliers_influence import variance_inflation_factor
import statsmodels.api as sm
import plotly.graph_objects as go

PNG_SCALE = 3  # ~300 dpi-equivalent on export (passed per-figure; works across kaleido versions)

# ---------- 0) Prep ----------
EXPOSURE_VARS = [
    "temperature_calibrated_median",
    "humidity_calibrated_median",
    "abs_humidity_c_median",
    "wet_bulb_temp_median",
]
LABELS = {
    "temperature_calibrated_median": "Temperature",
    "humidity_calibrated_median": "Relative humidity",
    "abs_humidity_c_median": "Absolute humidity",
    "wet_bulb_temp_median": "Wet bulb temperature",
}

REQ = ["Subject", "Room"] + EXPOSURE_VARS
missing = [c for c in REQ if c not in df_clean.columns]
assert not missing, f"Missing columns: {missing}"

df0 = df_clean[REQ].dropna().copy()
df0["Subject"] = df0["Subject"].astype("category")
df0["Room"] = df0["Room"].astype("category")

print(f"Analysis dataset -> Nights: {len(df0)}, Participants: {df0['Subject'].nunique()}")

# ---------- 1) Build within-person (person-demeaned) versions ----------
# This mirrors the "_within" covariate used in the actual OLS models, so the
# collinearity claim is checked on the SAME quantity the models regress on
# (raw pooled correlation mixes between-person and within-person variation).
g = df0.groupby("Subject", observed=True)
for v in EXPOSURE_VARS:
    df0[f"{v}_s"] = g[v].transform("median")
    df0[f"{v}_within"] = df0[v] - df0[f"{v}_s"]

within_cols = [f"{v}_within" for v in EXPOSURE_VARS]

# ---------- 2) Helper: pairwise Pearson r + 95% CI (Fisher z) + p ----------
def pairwise_corr_table(data, cols, labels):
    rows = []
    n = len(data)
    for i, a in enumerate(cols):
        for b in cols[i + 1:]:
            r, p = stats.pearsonr(data[a], data[b])
            # Fisher z CI
            z = np.arctanh(r)
            se = 1 / np.sqrt(n - 3)
            lo, hi = np.tanh(z - 1.96 * se), np.tanh(z + 1.96 * se)
            rows.append({
                "Variable 1": labels[a.replace("_within", "")] if a.replace("_within", "") in labels else a,
                "Variable 2": labels[b.replace("_within", "")] if b.replace("_within", "") in labels else b,
                "r": r,
                "95% CI": f"{lo:.2f} to {hi:.2f}",
                "p": p,
                "N": n,
            })
    return pd.DataFrame(rows)

pooled_tbl = pairwise_corr_table(df0, EXPOSURE_VARS, LABELS)
within_tbl = pairwise_corr_table(df0, within_cols, LABELS)

print("\n=== Pooled (raw) pairwise Pearson correlations ===")
print(pooled_tbl.round(3).to_string(index=False))

print("\n=== Within-person (person-demeaned) pairwise Pearson correlations ===")
print(within_tbl.round(3).to_string(index=False))

# ---------- 3) Variance Inflation Factors ----------
# VIF_j = 1 / (1 - R^2_j), where R^2_j comes from regressing variable j on the
# other three. VIF > 5 (conservative) or >10 (lenient) is the usual flag for
# "this variable's effect cannot be separated from the others."
def vif_table(data, cols, labels):
    X = sm.add_constant(data[cols])
    out = []
    for i, c in enumerate(cols):
        v = variance_inflation_factor(X.values, i + 1)  # +1 skips the constant column
        base = c.replace("_within", "")
        out.append({"Variable": labels.get(base, c), "VIF": v})
    return pd.DataFrame(out)

vif_pooled = vif_table(df0, EXPOSURE_VARS, LABELS)
vif_within = vif_table(df0, within_cols, LABELS)

print("\n=== VIF, pooled ===")
print(vif_pooled.round(2).to_string(index=False))

print("\n=== VIF, within-person ===")
print(vif_within.round(2).to_string(index=False))

# ---------- 4) Quantify "WBT is a deterministic function of T and RH" ----------
X = sm.add_constant(df0[["temperature_calibrated_median", "humidity_calibrated_median"]])
y = df0["wet_bulb_temp_median"]
r2_wbt = sm.OLS(y, X).fit().rsquared
print(f"\nR^2 of WBT ~ Temperature + Relative humidity (pooled): {r2_wbt:.4f}")

# ---------- 5) Build a single supplementary-ready table ----------
def star(p):
    return "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else ""

supp_tbl = pooled_tbl.copy()
supp_tbl["r (95% CI)"] = supp_tbl.apply(
    lambda row: f"{row['r']:.2f}{star(row['p'])} ({row['95% CI']})", axis=1
)
supp_tbl_within = within_tbl.copy()
supp_tbl_within["r_within (95% CI)"] = supp_tbl_within.apply(
    lambda row: f"{row['r']:.2f}{star(row['p'])} ({row['95% CI']})", axis=1
)

final_supp = supp_tbl[["Variable 1", "Variable 2", "r (95% CI)"]].merge(
    supp_tbl_within[["Variable 1", "Variable 2", "r_within (95% CI)"]],
    on=["Variable 1", "Variable 2"],
)
final_supp = final_supp.rename(columns={
    "r (95% CI)": "Pooled r (95% CI)",
    "r_within (95% CI)": "Within-person r (95% CI)",
})

# VIF panel: one row per variable (not per pair), pooled and within-person
# side by side, so the reader can see both the pairwise correlations AND the
# formal collinearity diagnostic in one supplementary table.
vif_panel = vif_pooled.rename(columns={"VIF": "Pooled VIF"}).merge(
    vif_within.rename(columns={"VIF": "Within-person VIF"}), on="Variable"
)
vif_panel["Pooled VIF"] = vif_panel["Pooled VIF"].round(1)
vif_panel["Within-person VIF"] = vif_panel["Within-person VIF"].round(1)

print("\n=== Supplementary Table: exposure-metric correlations and collinearity (VIF) ===")
print("\nPanel A — Pairwise Pearson correlations")
print(final_supp.to_string(index=False))
print("*p<0.05, **p<0.01, ***p<0.001 (pooled p-values are descriptive only; "
      "they do not account for repeated measures within participant / clustering — "
      "see notes)")
print("\nPanel B — Variance inflation factors (VIF ≈ 1/(1-R²) from regressing each "
      "variable on the other three; VIF > 5-10 flags severe collinearity)")
print(vif_panel.to_string(index=False))

# Single CSV with both panels stacked, so the whole supplementary table is one
# file: a title row for each panel, then that panel's table, blank line between.
with open(os.path.join(TABLES_DIR, "supp_table_exposure_correlations_and_vif.csv"), "w") as f:
    f.write("Panel A: Pairwise Pearson correlations among thermal exposure metrics\n")
    final_supp.to_csv(f, index=False)
    f.write("\nPanel B: Variance inflation factors (VIF)\n")
    vif_panel.to_csv(f, index=False)
_tbl_path = os.path.join(TABLES_DIR, "supp_table_exposure_correlations_and_vif.csv")
print(f"\nSaved: {_tbl_path}")

# ---------- 6) Heatmap figure for the supplement (Plotly) ----------
# Correlation is a POLARITY quantity (signed, centered on 0), not a plain
# magnitude, so it gets a diverging scale (blue <-> red, neutral gray at 0)
# rather than a single-hue sequential ramp.
corr_matrix = df0[EXPOSURE_VARS].corr(method="pearson")
corr_labels = [LABELS[c] for c in EXPOSURE_VARS]
corr_matrix.index = corr_matrix.columns = corr_labels

DIVERGING_SCALE = [
    [0.00, "#0d366b"],   # r = -1  (deep blue)
    [0.25, "#5598e7"],   # r = -0.5
    [0.50, "#f0efec"],   # r =  0  (neutral gray midpoint)
    [0.75, "#e8837a"],   # r = +0.5
    [1.00, "#8a2420"],   # r = +1  (deep red)
]

z = corr_matrix.values
n = len(corr_labels)
text = np.array([[f"{z[i, j]:.2f}" for j in range(n)] for i in range(n)])
# white text on the darker (high |r|) cells, ink text on the light/gray cells
text_color = np.where(np.abs(z) > 0.55, "#ffffff", "#0b0b0b")

fig = go.Figure(
    data=go.Heatmap(
        z=z,
        x=corr_labels,
        y=corr_labels,
        zmin=-1,
        zmax=1,
        colorscale=DIVERGING_SCALE,
        colorbar=dict(title="Pearson r", tickvals=[-1, -0.5, 0, 0.5, 1]),
        xgap=2,  # surface-gap between cells instead of a border stroke
        ygap=2,
    )
)
# Direct-labeled values on every cell (a small, fully-enumerable grid — this
# is the one case where labeling every mark is the right call, not the
# "never a number on every point" default for larger charts).
annotations = [
    dict(x=corr_labels[j], y=corr_labels[i], text=text[i, j], showarrow=False,
         font=dict(color=text_color[i, j], size=13))
    for i in range(n) for j in range(n)
]
fig.update_layout(
    title="Pairwise correlations among thermal exposure metrics",
    annotations=annotations,
    xaxis=dict(side="bottom", tickangle=-45, showgrid=False),
    yaxis=dict(autorange="reversed", showgrid=False),
    plot_bgcolor="#fcfcfb",
    paper_bgcolor="#fcfcfb",
    font=dict(family="system-ui, -apple-system, Segoe UI, sans-serif", color="#0b0b0b"),
    width=560,
    height=520,
    margin=dict(l=140, r=40, t=60, b=120),
)

fig.show()
# fig.write_image("supp_fig_exposure_correlation_heatmap.png", scale=PNG_SCALE)
# fig.write_image("supp_fig_exposure_correlation_heatmap.tiff", scale=PNG_SCALE * 2, format="tiff")  # 600 dpi TIFF (uncomment to export)
# fig.write_html("supp_fig_exposure_correlation_heatmap.html")  # interactive copy, optional
# print("Saved: supp_fig_exposure_correlation_heatmap.png (+ .html)")

# ---------- 7) Scatter matrix (Plotly SPLOM) ----------
# Pearson r only captures linear association, and WBT is a nonlinear function
# of T and RH — a scatter matrix is the visual companion that lets a reader
# see the actual shape of each pairwise relationship, not just its r.
# One relationship per panel (no grouping variable), so this is a single-hue
# mark, not a categorical palette: sequential blue at low opacity (a wash,
# not a saturated block), matching the "1-3 series -> color alone is fine"
# case with n=1 series here.
splom_dims = [dict(label=LABELS[v], values=df0[v]) for v in EXPOSURE_VARS]

fig2 = go.Figure(
    data=go.Splom(
        dimensions=splom_dims,
        showupperhalf=False,   # redundant with the lower triangle; decluttered
        diagonal_visible=False,
        marker=dict(
            color="#256abf",       # sequential blue, mid step
            size=4,                 # r ~ 4 -> >=8px diameter per mark spec
            opacity=0.25,            # wash, not a saturated block
            line=dict(width=0),
        ),
    )
)
fig2.update_layout(
    title="Pairwise relationships among thermal exposure metrics",
    plot_bgcolor="#fcfcfb",
    paper_bgcolor="#fcfcfb",
    font=dict(family="system-ui, -apple-system, Segoe UI, sans-serif", color="#0b0b0b"),
    width=750,
    height=750,
)
fig2.update_traces(diagonal_visible=False)
for ax in fig2.layout:
    if ax.startswith("xaxis") or ax.startswith("yaxis"):
        fig2.layout[ax].update(showgrid=True, gridcolor="#e1e0d9", gridwidth=1, zeroline=False)

fig2.show()
# fig2.write_image("supp_fig_exposure_scatter_matrix.png", scale=PNG_SCALE)
# fig2.write_image("supp_fig_exposure_scatter_matrix.tiff", scale=PNG_SCALE * 2, format="tiff")  # 600 dpi TIFF (uncomment to export)
# fig2.write_html("supp_fig_exposure_scatter_matrix.html")  # interactive copy, optional
# print("Saved: supp_fig_exposure_scatter_matrix.png (+ .html)")

Analysis dataset -> Nights: 694, Participants: 38

=== Pooled (raw) pairwise Pearson correlations ===
       Variable 1           Variable 2      r         95% CI     p   N
      Temperature    Relative humidity -0.606 -0.65 to -0.56 0.000 694
      Temperature    Absolute humidity  0.454   0.39 to 0.51 0.000 694
      Temperature Wet bulb temperature  0.747   0.71 to 0.78 0.000 694
Relative humidity    Absolute humidity  0.407   0.34 to 0.47 0.000 694
Relative humidity Wet bulb temperature  0.054  -0.02 to 0.13 0.152 694
Absolute humidity Wet bulb temperature  0.930   0.92 to 0.94 0.000 694

=== Within-person (person-demeaned) pairwise Pearson correlations ===
       Variable 1           Variable 2      r         95% CI   p   N
      Temperature    Relative humidity -0.394 -0.46 to -0.33 0.0 694
      Temperature    Absolute humidity  0.338   0.27 to 0.40 0.0 694
      Temperature Wet bulb temperature  0.603   0.55 to 0.65 0.0 694
Relative humidity    Absolute humidity  0.668   0.62 t

## 8. Descriptive Statistics for Nighttime Sleep Periods — Table 1 (n=694 nights)

In [18]:

# ---- Helpers ----
def mean_iqr(series):
    v = series.dropna().to_numpy()
    return np.nanmean(v), np.nanpercentile(v,25), np.nanpercentile(v,75)

def mean_std(series):
    v = series.dropna().to_numpy()
    return np.nanmean(v), np.nanstd(v)

def median_iqr(series):
    v = series.dropna().to_numpy()
    return np.nanmedian(v), np.nanpercentile(v,25), np.nanpercentile(v,75)

def min_max(series):
    v = series.dropna().to_numpy()
    return np.nanmin(v), np.nanmax(v)

def fmt_num(x, k=2):
    return f"{x:.{k}f}"

def fmt_range(lo, hi, k=2):
    return f"{fmt_num(lo,k)}–{fmt_num(hi,k)}"

def median_p5_p95(series):
    v = series.dropna().to_numpy()
    return np.nanmedian(v), np.nanpercentile(v,5), np.nanpercentile(v,95)

# ---- Counts ----
n_rooms        = dc["Room"].nunique() if "Room" in dc.columns else np.nan
n_participants = dc["Subject"].nunique()
n_nights       = len(dc)
nights_per_subj_mean = dc.groupby("Subject", observed=True)["TimeAsleepInMinutes"].size().mean()
# Nights per participant: mean and range
nights_per_subj_counts = dc.groupby("Subject", observed=True)["TimeAsleepInMinutes"].size()
nights_per_subj_min = nights_per_subj_counts.min()
nights_per_subj_max = nights_per_subj_counts.max()


subject_ids = sorted(dc["Subject"].dropna().unique())

# ---- Environment summaries ----
# Mean and SD
temp_mean, temp_std = mean_std(dc["temperature_calibrated_mean"])
temp_outdoor_mean, temp_outdoor_std = mean_std(dc["temperature_calibrated_mean_Outdoor"])
rh_mean,   rh_std   = mean_std(dc["humidity_calibrated_mean"])
ah_c_mean,   ah_c_std   = mean_std(dc["abs_humidity_c_mean"])
wet_mean, wet_std = mean_std(dc["wet_bulb_temp_mean"])
co2_mean,  co2_std = mean_std(dc["co2_used_mean"])
pm25_mean, pm25_std = mean_std(dc["pm25_mean"])
pm1_mean, pm1_std = mean_std(dc["pm1_mean"])
pm10_mean, pm10_std = mean_std(dc["pm10_mean"])

# Median and IQR
temp_med, temp_q1, temp_q3 = median_iqr(dc["temperature_calibrated_median"])
temp_outdoor_med, temp_outdoor_q1, temp_outdoor_q3 = median_iqr(dc["temperature_calibrated_median_Outdoor"])
rh_med,   rh_q1,   rh_q3   = median_iqr(dc["humidity_calibrated_median"])
ah_c_med,   ah_c_q1,   ah_c_q3   = median_iqr(dc["abs_humidity_c_median"])
wet_med, wet_q1, wet_q3 = median_iqr(dc["wet_bulb_temp_median"])
co2_med,  co2_q1,  co2_q3  = median_iqr(dc["co2_used_median"])
pm25_med, pm25_q1, pm25_q3 = median_iqr(dc["pm25_median"])
pm1_med, pm1_q1, pm1_q3 = median_iqr(dc["pm1_median"])
pm10_med, pm10_q1, pm10_q3 = median_iqr(dc["pm10_median"])

# Median of the medians + 5th/95th
temp_med, temp_p5, temp_p95 = median_p5_p95(dc["temperature_calibrated_median"])
temp_outdoor_med, temp_outdoor_p5, temp_outdoor_p95 = median_p5_p95(dc["temperature_calibrated_median_Outdoor"])
rh_med,   rh_p5,   rh_p95   = median_p5_p95(dc["humidity_calibrated_median"])
ah_c_med,   ah_c_p5,   ah_c_p95   = median_p5_p95(dc["abs_humidity_c_median"])
wet_med, wet_p5, wet_p95 = median_p5_p95(dc["wet_bulb_temp_median"])
co2_med,  co2_p5,  co2_p95  = median_p5_p95(dc["co2_used_median"])
pm25_med, pm25_p5, pm25_p95 = median_p5_p95(dc["pm25_median"])
pm1_med, pm1_p5, pm1_p95    = median_p5_p95(dc["pm1_median"])
pm10_med, pm10_p5, pm10_p95 = median_p5_p95(dc["pm10_median"])

# Min / Max
temp_min, temp_max = min_max(dc["temperature_calibrated_median"])
temp_outdoor_min, temp_outdoor_max = min_max(dc["temperature_calibrated_median_Outdoor"])
rh_min,   rh_max   = min_max(dc["humidity_calibrated_median"])
ah_c_min,   ah_c_max   = min_max(dc["abs_humidity_c_median"])
wet_min, wet_max = min_max(dc["wet_bulb_temp_median"])
co2_min,  co2_max  = min_max(dc["co2_used_median"])
pm25_min, pm25_max = min_max(dc["pm25_median"])
pm1_min, pm1_max = min_max(dc["pm1_median"])
pm10_min, pm10_max = min_max(dc["pm10_median"])

# ---- Sleep summaries ----
# Mean and SD
tst_mean, tst_std = mean_std(dc["TimeAsleepInMinutes"])
sol_mean, sol_std = mean_std(dc["LatencyInMinutes"])
se_mean, se_std   = mean_std(dc["Efficiency"])
se_calc_mean, se_calc_std = mean_std(dc["SE_recalculated"])
waso_mean, waso_std = mean_std(dc["WakeAfterOnsetInMinutes"])
tib_mean, tib_std = mean_std(dc["TotalInBedTime_minutes"])
tib_calc_mean, tib_calc_std = mean_std(dc["TIB_recalculated"])
awakening_mean, awakening_std = mean_std(dc["AwakeningCount"])

# Median and IQR
tst_med,  tst_q1,  tst_q3  = median_iqr(dc["TimeAsleepInMinutes"])
sol_med,  sol_q1,  sol_q3  = median_iqr(dc["LatencyInMinutes"]) 
se_med,   se_q1,   se_q3   = median_iqr(dc["Efficiency"]) 
se_calc_med,   se_calc_q1,   se_calc_q3   = median_iqr(dc["SE_recalculated"])
waso_med, waso_q1, waso_q3 = median_iqr(dc["WakeAfterOnsetInMinutes"]) 
tib_med, tib_q1, tib_q3 = median_iqr(dc["TotalInBedTime_minutes"]) 
tib_calc_med, tib_calc_q1, tib_calc_q3 = median_iqr(dc["TIB_recalculated"]) 
awakening_med, awakening_q1, awakening_q3 = median_iqr(dc["AwakeningCount"])

# Median + 5th/95th
tst_med,  tst_p5,  tst_p95  = median_p5_p95(dc["TimeAsleepInMinutes"])
sol_med,  sol_p5,  sol_p95  = median_p5_p95(dc["LatencyInMinutes"])
se_med,   se_p5,   se_p95   = median_p5_p95(dc["Efficiency"]) 
se_calc_med, se_calc_p5, se_calc_p95 = median_p5_p95(dc["SE_recalculated"]) 
waso_med, waso_p5, waso_p95 = median_p5_p95(dc["WakeAfterOnsetInMinutes"])
tib_med, tib_p5, tib_p95 = median_p5_p95(dc["TotalInBedTime_minutes"]) 
tib_calc_med, tib_calc_p5, tib_calc_p95 = median_p5_p95(dc["TIB_recalculated"])
awakening_med, awakening_p5, awakening_p95 = median_p5_p95(dc["AwakeningCount"])

# Min / Max
tst_min,  tst_max  = min_max(dc["TimeAsleepInMinutes"])
sol_min,  sol_max  = min_max(dc["LatencyInMinutes"]) if "LatencyInMinutes" in dc.columns else (np.nan,np.nan)
se_min,   se_max   = min_max(dc["Efficiency"]) if "Efficiency" in dc.columns else (np.nan,np.nan)
se_calc_min,   se_calc_max   = min_max(dc["SE_recalculated"]) if "SE_recalculated" in dc.columns else (np.nan,np.nan)
waso_min, waso_max = min_max(dc["WakeAfterOnsetInMinutes"]) if "WakeAfterOnsetInMinutes" in dc.columns else (np.nan,np.nan)
tib_min, tib_max = min_max(dc["TotalInBedTime_minutes"]) if "TotalInBedTime_minutes" in dc.columns else (np.nan,np.nan)
tib_calc_min, tib_calc_max = min_max(dc["TIB_recalculated"]) if "TIB_recalculated" in dc.columns else (np.nan,np.nan)
awakening_min, awakening_max = min_max(dc["AwakeningCount"]) if "AwakeningCount" in dc.columns else (np.nan,np.nan)

# # If efficiency appears to be 0–1, convert to %
# if np.nanmedian(dc["Efficiency"]) is not np.nan and np.nanmedian(dc["Efficiency"]) <= 2:
#     se_med, se_q1, se_q3 = se_med*100, se_q1*100, se_q3*100
#     se_min, se_max = se_min*100, se_max*100

# ---- Build paste-ready markdown (unchanged unless min/max needs to be included) ----
md = f"""
Across {n_nights} nights (≈ {nights_per_subj_mean:.1f} per participant; range **{nights_per_subj_min}–{nights_per_subj_max}) from {n_participants}** participants residing in {n_rooms} rooms, we observed a 
Mean bedroom temperature of {fmt_num(temp_mean,2)} °C (IQR {fmt_range(temp_q1, temp_q3,2)}), and relative humidity of {fmt_num(rh_mean,0)}% (IQR {fmt_num(rh_q1,0)}–{fmt_num(rh_q3,0)}). 
Median sleep metrics were: TST {fmt_num(tst_med,0)} min (IQR {fmt_num(tst_q1,0)}–{fmt_num(tst_q3,0)}), SOL {fmt_num(sol_med,0)} min (IQR {fmt_num(sol_q1,0)}–{fmt_num(sol_q3,0)}), SE {fmt_num(se_calc_med,0)}% (IQR {fmt_num(se_calc_q1,0)}–{fmt_num(se_calc_q3,0)}), and WASO {fmt_num(waso_med,0)} min (IQR {fmt_num(waso_q1,0)}–{fmt_num(waso_q3,0)}). 
Mean sleep metrics were: TST {fmt_num(tst_mean,0)} min (SD \u00B1 {fmt_num(tst_std,0)}), SOL {fmt_num(sol_mean,0)} min (SD \u00B1 {fmt_num(sol_std,0)}), SE {fmt_num(se_calc_mean,0)}% (SD \u00B1 {fmt_num(se_calc_std,0)}), and WASO {fmt_num(waso_mean,0)} min (SD \u00B1 {fmt_num(waso_std,0)}). 
""".strip()

print("\n— COUNTS —")
print(f"Participants: {n_participants} | Rooms: {n_rooms} | Nights: {n_nights} | Nights/participant (mean): {nights_per_subj_mean:.1f}")
print(f"Nights/participant range: {nights_per_subj_min}–{nights_per_subj_max}")
print("Subject IDs:", subject_ids)


# print("\n— ENVIRONMENT means—")
# print(f"Temperature mean {temp_mean:.2f} °C (IQR {temp_q1:.2f}–{temp_q3:.2f}) | Min–Max {temp_min:.2f}–{temp_max:.2f}")
# print(f"Relative humidity mean {rh_mean:.0f}% (IQR {rh_q1:.0f}–{rh_q3:.0f}) | Min–Max {rh_min:.0f}–{rh_max:.0f}")
# print(f"CO2 mean {co2_mean:.0f} ppm (IQR {co2_q1:.0f}–{co2_q3:.0f}) | Min–Max {co2_min:.0f}–{co2_max:.0f}")
# print(f"PM2.5 mean {pm25_mean:.0f} ? (IQR {pm25_q1:.0f}–{pm25_q3:.0f}) | Min–Max {pm25_min:.0f}–{pm25_max:.0f}")
# print(f"PM1.0 mean {pm1_mean:.0f} ? (IQR {pm1_q1:.0f}–{pm1_q3:.0f}) | Min–Max {pm1_min:.0f}–{pm1_max:.0f}")
# print(f"PM10 mean {pm10_mean:.0f} ? (IQR {pm10_q1:.0f}–{pm10_q3:.0f}) | Min–Max {pm10_min:.0f}–{pm10_max:.0f}")


# ---- Printout with 5th–95th percentiles ----
print("\n— FOR TABLE (medians, IQR, 5-95th Percentile), Min-Max —")
print("\n— Nighttime Indoor Environmental Conditions —")
print(f"Temperature median {temp_med:.2f} °C | IQR {temp_q1:.2f}–{temp_q3:.2f} | 5–95th: {temp_p5:.2f}–{temp_p95:.2f} |Min–Max {temp_min:.2f}–{temp_max:.2f}")
# print(f"Outdoor temperature median {temp_outdoor_med:.2f} °C | IQR {temp_outdoor_q1:.2f}–{temp_outdoor_q3:.2f} | 5–95th: {temp_outdoor_p5:.2f}–{temp_outdoor_p95:.2f} | Min–Max {temp_outdoor_min:.2f}–{temp_outdoor_max:.2f}")
print(f"Absolute humidity median {ah_c_med:.0f} g/m3 | IQR {ah_c_q1:.0f}–{ah_c_q3:.0f} | 5–95th: {ah_c_p5:.0f}–{ah_c_p95:.0f} | Min–Max {ah_c_min:.0f}–{ah_c_max:.0f}")
print(f"Relative humidity median {rh_med:.0f}% | IQR {rh_q1:.0f}–{rh_q3:.0f} | 5–95th: {rh_p5:.0f}–{rh_p95:.0f} | Min–Max {rh_min:.0f}–{rh_max:.0f}")
print(f"Wet bulb temperature median {wet_med:.2f} °C | IQR {wet_q1:.2f}–{wet_q3:.2f} | 5–95th: {wet_p5:.2f}–{wet_p95:.2f} | Min–Max {wet_min:.2f}–{wet_max:.2f}")
print(f"CO2 median {co2_med:.0f} ppm | IQR {co2_q1:.0f}–{co2_q3:.0f} | 5–95th: {co2_p5:.0f}–{co2_p95:.0f} | Min–Max {co2_min:.0f}–{co2_max:.0f}")
print(f"PM2.5 median {pm25_med:.0f} μg/m3 | IQR {pm25_q1:.0f}–{pm25_q3:.0f} | 5–95th: {pm25_p5:.0f}–{pm25_p95:.0f} | Min–Max {pm25_min:.0f}–{pm25_max:.0f}")

print("\n— Sleep Outcomes —")
print(f"TIB_recalc median {tib_calc_med:.2f} min | IQR {tib_calc_q1:.2f}–{tib_calc_q3:.2f} | 5–95th: {tib_calc_p5:.2f}–{tib_calc_p95:.2f} | Min–Max {tib_calc_min:.2f}–{tib_calc_max:.2f}")
print(f"TST median {tst_med:.0f} min | IQR {tst_q1:.2f}–{tst_q3:.2f} | 5–95th: {tst_p5:.0f}–{tst_p95:.0f} | Min–Max {tst_min:.2f}–{tst_max:.2f}")
print(f"SE median {se_calc_med:.0f}% | IQR {se_calc_q1:.0f}–{se_calc_q3:.0f}% | 5–95th: {se_calc_p5:.0f}–{se_calc_p95:.0f} | Min–Max {se_calc_min:.0f}–{se_calc_max:.0f}%")
print(f"SOL median {sol_med:.0f} min | IQR {sol_q1:.0f}–{sol_q3:.0f} | 5–95th: {sol_p5:.0f}–{sol_p95:.0f} | Min–Max {sol_min:.0f}–{sol_max:.0f}")
print(f"WASO median {waso_med:.0f} min | IQR {waso_q1:.0f}–{waso_q3:.0f} | 5–95th: {waso_p5:.0f}–{waso_p95:.0f} | Min–Max {waso_min:.0f}–{waso_max:.0f}")
print(f"Awakenings median {awakening_med:.0f} | IQR {awakening_q1:.0f}–{awakening_q3:.0f} | 5–95th: {awakening_p5:.0f}–{awakening_p95:.0f} | Min–Max {awakening_min:.0f}–{awakening_max:.0f}")

print("\n" +md)



— COUNTS —
Participants: 38 | Rooms: 8 | Nights: 694 | Nights/participant (mean): 18.3
Nights/participant range: 8–33
Subject IDs: ['CDS019', 'CDS026', 'CDS027', 'CDS029', 'CDS030', 'CDS032', 'CDS035', 'CDS037', 'CDS040', 'CDS101', 'CDS102', 'CDS103', 'CDS104', 'CDS105', 'CDS106', 'CDS107', 'CDS108', 'CDS109', 'CDS111', 'CDS112', 'CDS113', 'CDS114', 'CDS115', 'CDS116', 'CDS117', 'CDS118', 'CDS119', 'CDS120', 'CDS121', 'CDS122', 'CDS123', 'CDS124', 'CDS126', 'CDS128', 'CDS129', 'CDS130', 'CDS131', 'CDS132']

— FOR TABLE (medians, IQR, 5-95th Percentile), Min-Max —

— Nighttime Indoor Environmental Conditions —
Temperature median 30.55 °C | IQR 30.06–31.36 | 5–95th: 29.03–32.75 |Min–Max 27.97–33.60
Absolute humidity median 24 g/m3 | IQR 23–25 | 5–95th: 22–26 | Min–Max 20–27
Relative humidity median 76% | IQR 73–79 | 5–95th: 68–81 | Min–Max 63–86
Wet bulb temperature median 27.14 °C | IQR 26.59–27.72 | 5–95th: 25.95–28.55 | Min–Max 25.13–29.53
CO2 median 629 ppm | IQR 580–737 | 5–95th: 5

In [19]:
# ---- % of nights with xx ----

tst_threshold = 420
sleep_col = "TimeAsleepInMinutes" 

n_nights_total = len(df_clean)  # should be 694
n_nights_valid = df_clean[sleep_col].notna().sum()

n_nights_above_x = (df_clean[sleep_col] > tst_threshold).sum()
pct_nights_above_x = (n_nights_above_x / n_nights_total) * 100

print("\n— TST THRESHOLD —")
print(
    f"Nights with TST more than 7h > {tst_threshold}: "
    f"{n_nights_above_x}/{n_nights_total} "
    f"({pct_nights_above_x:.1f}%)"
)


— TST THRESHOLD —
Nights with TST more than 7h > 420: 39/694 (5.6%)


In [20]:
# ---- % of nights with xx ----

se_threshold = 85
sleep_col = "SE_recalculated" 

n_nights_total = len(df_clean)  # should be 694
n_nights_valid = df_clean[sleep_col].notna().sum()

n_nights_above_x = (df_clean[sleep_col] > se_threshold).sum()
pct_nights_above_x = (n_nights_above_x / n_nights_total) * 100

print("\n— SE THRESHOLD —")
print(
    f"Nights with SE more than 85% > {se_threshold}: "
    f"{n_nights_above_x}/{n_nights_total} "
    f"({pct_nights_above_x:.1f}%)"
)


— SE THRESHOLD —
Nights with SE more than 85% > 85: 126/694 (18.2%)


## 9. Primary Analysis — Within-Subject OLS Regressions (Table S1 / Figure 2)

For each of six environmental exposures and five sleep outcomes, fits

```
outcome ~ (exposure − subject's median exposure) / scale + C(Room)
```

with subject-clustered standard errors, where `scale` rescales CO₂ to "per 100 ppm" and
PM₂.₅ to "per 10 µg/m³" increments (all other exposures use their native 1-unit increment).
Room fixed effects control for room-level characteristics; centering each night's exposure
on the participant's own median isolates the within-subject effect. P-values are
Bonferroni-corrected across the five outcomes, separately within each exposure.

In [21]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests

EXPOSURES = {
    "Temperature": dict(col="temperature_calibrated_median", scale=1, unit="1 °C"),
    "Absolute Humidity": dict(col="abs_humidity_c_median", scale=1, unit="1 g/m³"),
    "Relative Humidity": dict(col="humidity_calibrated_median", scale=1, unit="1%"),
    "Wet Bulb Temperature": dict(col="wet_bulb_temp_median", scale=1, unit="1 °C"),
    "CO2": dict(col="co2_used_median", scale=100, unit="100 ppm"),
    "PM2.5": dict(col="pm25_median", scale=10, unit="10 μg/m³"),
}

OUTCOMES = {
    "TIB": "TIB_recalculated",
    "TST": "TimeAsleepInMinutes",
    "SE": "SE_recalculated",
    "SOL": "LatencyInMinutes",
    "WASO": "WakeAfterOnsetInMinutes",
}

NIGHT_COL = "Adjusted_date"  # per-night date, used later for residual autocorrelation checks


def fit_exposure_models(exposure_col, scale):
    """Fit all 5 outcome models for one environmental exposure."""
    req = ["Subject", "Room", NIGHT_COL, exposure_col] + list(OUTCOMES.values())
    df0 = df_clean[req].dropna().copy()
    df0["Subject"] = df0["Subject"].astype("category")
    df0["Room"] = df0["Room"].astype("category")

    within_col = f"{exposure_col}_within"
    subject_median = df0.groupby("Subject", observed=True)[exposure_col].transform("median")
    df0[within_col] = (df0[exposure_col] - subject_median) / scale

    def fit_outcome(outcome_col):
        formula = f"{outcome_col} ~ {within_col} + C(Room)"
        return smf.ols(formula, data=df0).fit(
            cov_type="cluster",
            cov_kwds={"groups": df0["Subject"], "use_correction": True, "df_correction": True},
        )

    models = {outcome_name: fit_outcome(outcome_col) for outcome_name, outcome_col in OUTCOMES.items()}
    return df0, within_col, models


ols_results = {}
for exposure_name, spec in EXPOSURES.items():
    print(f"\n=== {exposure_name} ({spec['unit']} increment) ===")
    df0, within_col, models = fit_exposure_models(spec["col"], spec["scale"])
    print(f"Analysis dataset -> Nights: {len(df0)}, Participants: {df0['Subject'].nunique()}")

    p_raw = [models[outcome].pvalues[within_col] for outcome in OUTCOMES]
    _, p_bonf, _, _ = multipletests(p_raw, alpha=0.05, method="bonferroni")

    rows = []
    for outcome_name, p_adj in zip(OUTCOMES, p_bonf):
        m = models[outcome_name]
        ci = m.conf_int()
        beta = m.params[within_col]
        lo, hi = ci.loc[within_col, 0], ci.loc[within_col, 1]
        p_this = m.pvalues[within_col]
        rows.append(dict(Outcome=outcome_name, beta=beta, ci_lo=lo, ci_hi=hi, p_raw=p_this, p_bonferroni=p_adj))
        print(f"  {outcome_name}: beta={beta:+.2f} (95% CI {lo:+.2f} to {hi:+.2f}; "
              f"raw p={p_this:.3f}; Bonferroni p={p_adj:.3f})")

    ols_results[exposure_name] = dict(
        column=spec["col"], scale=spec["scale"], unit=spec["unit"],
        df=df0, within_col=within_col, models=models,
        summary=pd.DataFrame(rows).set_index("Outcome"),
    )

print("\nFull statsmodels output for any (exposure, outcome) pair is available via, e.g.:")
print('  ols_results["Temperature"]["models"]["TST"].summary()')


=== Temperature (1 °C increment) ===
Analysis dataset -> Nights: 694, Participants: 38
  TIB: beta=-6.39 (95% CI -12.67 to -0.11; raw p=0.046; Bonferroni p=0.231)
  TST: beta=-12.53 (95% CI -17.48 to -7.58; raw p=0.000; Bonferroni p=0.000)
  SE: beta=-1.76 (95% CI -2.50 to -1.01; raw p=0.000; Bonferroni p=0.000)
  SOL: beta=+0.22 (95% CI -1.06 to +1.51; raw p=0.735; Bonferroni p=1.000)
  WASO: beta=+5.92 (95% CI +1.93 to +9.91; raw p=0.004; Bonferroni p=0.018)

=== Absolute Humidity (1 g/m³ increment) ===
Analysis dataset -> Nights: 694, Participants: 38
  TIB: beta=-0.87 (95% CI -5.18 to +3.45; raw p=0.694; Bonferroni p=1.000)
  TST: beta=-4.22 (95% CI -6.97 to -1.48; raw p=0.003; Bonferroni p=0.013)
  SE: beta=-0.76 (95% CI -1.29 to -0.24; raw p=0.004; Bonferroni p=0.022)
  SOL: beta=+0.41 (95% CI -0.50 to +1.32; raw p=0.373; Bonferroni p=1.000)
  WASO: beta=+2.95 (95% CI -0.16 to +6.05; raw p=0.063; Bonferroni p=0.316)

=== Relative Humidity (1% increment) ===
Analysis dataset -> N

### Residual Diagnostics

In [22]:
# =====================================================================
# Residual diagnostics across all 5 OLS outcome models, for each of the 6
# exposures (normality + within-participant autocorrelation across
# consecutive nights).
# =====================================================================

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import shapiro, norm, probplot
from statsmodels.stats.stattools import durbin_watson
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tsa.stattools import acf


COLOR_DATA = "#2a78d6"
COLOR_REF = "#898781"
COLOR_GRID = "#e1e0d9"
COLOR_TEXT = "#52514e"

EXPOSURES = [
    "Temperature", "Absolute Humidity", "Relative Humidity",
    "Wet Bulb Temperature", "CO2", "PM2.5",
]

for EXPOSURE_NAME in EXPOSURES:
    exposure_slug = EXPOSURE_NAME.replace(" ", "_")
    print("\n" + "=" * 70)
    print(f"RESIDUAL DIAGNOSTICS — {EXPOSURE_NAME}")
    print("=" * 70)

    df0 = ols_results[EXPOSURE_NAME]["df"]
    models = ols_results[EXPOSURE_NAME]["models"]
    night_col = NIGHT_COL

    # =================================================================
    # 1) Residual normality — per outcome
    # =================================================================
    norm_rows = []
    resids = {}
    for name, m in models.items():
        r = m.resid.dropna()
        resids[name] = r
        W, p_shapiro = shapiro(r.to_numpy())
        norm_rows.append({
            "Outcome": name,
            "n": len(r),
            "Shapiro W": W,
            "Shapiro p": p_shapiro,
            "Skew": r.skew(),
            "Excess kurtosis": r.kurt(),
        })

    norm_summary = pd.DataFrame(norm_rows).sort_values("Shapiro W")
    print(f"— Residual normality by outcome, {EXPOSURE_NAME} (worst first) —")
    print(norm_summary.to_string(index=False, float_format=lambda x: f"{x:.3f}"))

    # -- Grid of histogram + Q–Q per outcome --
    outcome_order = norm_summary["Outcome"].tolist()  # worst-normality first
    fig1 = make_subplots(
        rows=len(outcome_order), cols=2,
        subplot_titles=sum([[f"{o} — histogram", f"{o} — Q–Q"] for o in outcome_order], []),
        vertical_spacing=0.04,
    )

    for i, name in enumerate(outcome_order, start=1):
        r = resids[name].to_numpy()

        counts, bin_edges = np.histogram(r, bins=30)
        bin_width = bin_edges[1] - bin_edges[0]
        fig1.add_trace(
            go.Bar(x=(bin_edges[:-1] + bin_edges[1:]) / 2, y=counts, width=bin_width,
                   marker_color=COLOR_DATA, marker_line_width=0, showlegend=False,
                   hovertemplate="residual ≈ %{x:.2f}<br>count = %{y}<extra></extra>"),
            row=i, col=1,
        )
        x_grid = np.linspace(r.min(), r.max(), 200)
        normal_curve = norm.pdf(x_grid, r.mean(), r.std()) * len(r) * bin_width
        fig1.add_trace(
            go.Scatter(x=x_grid, y=normal_curve, mode="lines",
                       line=dict(color=COLOR_REF, width=2, dash="dash"),
                       showlegend=False, hoverinfo="skip"),
            row=i, col=1,
        )

        (osm, osr), (slope, intercept, r_fit) = probplot(r, dist="norm")
        fig1.add_trace(
            go.Scatter(x=osm, y=osr, mode="markers",
                       marker=dict(color=COLOR_DATA, size=5),
                       showlegend=False,
                       hovertemplate="theoretical = %{x:.2f}<br>sample = %{y:.2f}<extra></extra>"),
            row=i, col=2,
        )
        qq_x = np.array([osm.min(), osm.max()])
        fig1.add_trace(
            go.Scatter(x=qq_x, y=slope * qq_x + intercept, mode="lines",
                       line=dict(color=COLOR_REF, width=2, dash="dash"),
                       showlegend=False, hoverinfo="skip"),
            row=i, col=2,
        )
        fig1.update_xaxes(gridcolor=COLOR_GRID, row=i, col=1)
        fig1.update_yaxes(gridcolor=COLOR_GRID, row=i, col=1)
        fig1.update_xaxes(gridcolor=COLOR_GRID, row=i, col=2)
        fig1.update_yaxes(gridcolor=COLOR_GRID, row=i, col=2)

    fig1.update_layout(
        template="plotly_white", height=280 * len(outcome_order), width=920,
        font=dict(color=COLOR_TEXT),
        margin=dict(t=50, l=60, r=20, b=40),
        title=f"Residual normality — {EXPOSURE_NAME} — all outcomes (worst Shapiro-Wilk W first)",
    )
    # fig1.write_html(f"residual_normality_{exposure_slug}.html", include_plotlyjs="cdn")
    # fig1.write_image(os.path.join(FIGURES_DIR, f"residual_normality_{exposure_slug}.tiff"), scale=6, format="tiff")  # 600 dpi TIFF (uncomment to export)
    fig1.show()

    # =================================================================
    # 2) Autocorrelation across consecutive nights — per outcome
    # =================================================================
    missing = [c for c in ("Subject", night_col) if c not in df0.columns]
    if missing:
        hint = [c for c in df0.columns if any(k in c.lower() for k in ("night", "date", "day"))]
        print(
            f"\n[Skipping autocorrelation section for {EXPOSURE_NAME}] {missing} not in df0.columns.\n"
            f"Add a per-subject night/date column to `REQ` in OLS block, "
            f"then set `night_col` above to its name.\n"
            f"Columns that look like candidates: {hint}\n"
            f"All df0 columns: {list(df0.columns)}"
        )
        continue

    def lag1_autocorr(s):
        s = s.dropna()
        return s.autocorr(lag=1) if len(s) > 2 else np.nan

    ac_rows = []
    acf_by_outcome = {}
    for name, m in models.items():
        r = m.resid
        d = df0.loc[r.index, ["Subject", night_col]].copy()
        d["resid"] = r.values
        d = d.sort_values(["Subject", night_col])

        dw = durbin_watson(d["resid"].dropna())

        ac1_by_subj = d.groupby("Subject", observed=True)["resid"].apply(lag1_autocorr)

        lb_pvals = []
        for _, g in d.groupby("Subject", observed=True):
            rr = g["resid"].dropna()
            if len(rr) > 5:
                lb = acorr_ljungbox(rr, lags=[1], return_df=True)
                lb_pvals.append(lb["lb_pvalue"].iloc[0])
        lb_pvals = np.array(lb_pvals)
        pct_sig = 100 * np.mean(lb_pvals < 0.05) if len(lb_pvals) else np.nan

        ac_rows.append({
            "Outcome": name,
            "Durbin-Watson (global)": dw,
            "Mean lag-1 autocorr": ac1_by_subj.mean(),
            "Min lag-1 autocorr": ac1_by_subj.min(),
            "Max lag-1 autocorr": ac1_by_subj.max(),
            "% subjects Ljung-Box sig (p<.05)": pct_sig,
            "n subjects tested": len(lb_pvals),
        })

        resid_series = d["resid"].dropna().to_numpy()
        n_lags = 10
        acf_by_outcome[name] = acf(resid_series, nlags=n_lags, fft=True)

    ac_summary = pd.DataFrame(ac_rows).sort_values("Mean lag-1 autocorr", ascending=False)
    print(f"\n— Autocorrelation across consecutive nights, {EXPOSURE_NAME}, by outcome —")
    print(ac_summary.to_string(index=False, float_format=lambda x: f"{x:.3f}"))

    # -- ACF bars, one panel per outcome --
    fig2 = make_subplots(rows=1, cols=len(models), subplot_titles=list(models.keys()),
                          shared_yaxes=True)
    n_obs_by_outcome = {name: len(resids[name]) for name in models}
    for i, name in enumerate(models.keys(), start=1):
        vals = acf_by_outcome[name]
        lags = np.arange(len(vals))
        ci_bound = 1.96 / np.sqrt(n_obs_by_outcome[name])
        fig2.add_shape(type="rect", x0=-0.5, x1=len(vals) - 0.5, y0=-ci_bound, y1=ci_bound,
                        fillcolor=COLOR_GRID, opacity=0.6, line_width=0, layer="below",
                        row=1, col=i)
        fig2.add_trace(
            go.Bar(x=lags, y=vals, marker_color=COLOR_DATA, width=0.3, showlegend=False,
                   hovertemplate="lag = %{x}<br>ACF = %{y:.3f}<extra></extra>"),
            row=1, col=i,
        )
        fig2.update_xaxes(title_text="Lag", dtick=1, gridcolor=COLOR_GRID, row=1, col=i)
    fig2.update_yaxes(title_text="Autocorrelation", gridcolor=COLOR_GRID, row=1, col=1)
    fig2.update_layout(
        template="plotly_white", height=380, width=1100,
        font=dict(color=COLOR_TEXT),
        margin=dict(t=60, l=60, r=20, b=50),
        title=f"Residual ACF by outcome — {EXPOSURE_NAME} (subject-ordered; shaded = 95% band)",
    )
    # fig2.write_html(f"residual_acf_{exposure_slug}.html", include_plotlyjs="cdn")
    # fig2.write_image(os.path.join(FIGURES_DIR, f"residual_acf_{exposure_slug}.tiff"), scale=6, format="tiff")  # 600 dpi TIFF (uncomment to export)
    fig2.show()


RESIDUAL DIAGNOSTICS — Temperature
— Residual normality by outcome, Temperature (worst first) —
Outcome   n  Shapiro W  Shapiro p   Skew  Excess kurtosis
    SOL 694      0.840      0.000  1.693            3.009
   WASO 694      0.986      0.000  0.461            0.227
    TIB 694      0.991      0.000 -0.183            0.707
    TST 694      0.996      0.091 -0.022            0.472
     SE 694      0.996      0.103 -0.204           -0.120



— Autocorrelation across consecutive nights, Temperature, by outcome —
Outcome  Durbin-Watson (global)  Mean lag-1 autocorr  Min lag-1 autocorr  Max lag-1 autocorr  % subjects Ljung-Box sig (p<.05)  n subjects tested
    TIB                   1.291                0.000              -0.508               0.474                             5.263                 38
   WASO                   1.303               -0.026              -0.352               0.643                             2.632                 38
    TST                   1.187               -0.046              -0.786               0.465                             5.263                 38
     SE                   1.214               -0.062              -0.651               0.542                             7.895                 38
    SOL                   1.851               -0.078              -0.434               0.348                             0.000                 38



RESIDUAL DIAGNOSTICS — Absolute Humidity
— Residual normality by outcome, Absolute Humidity (worst first) —
Outcome   n  Shapiro W  Shapiro p   Skew  Excess kurtosis
    SOL 694      0.840      0.000  1.697            3.022
   WASO 694      0.986      0.000  0.442            0.118
    TIB 694      0.991      0.000 -0.173            0.732
     SE 694      0.995      0.015 -0.214           -0.187
    TST 694      0.996      0.097 -0.043            0.417



— Autocorrelation across consecutive nights, Absolute Humidity, by outcome —
Outcome  Durbin-Watson (global)  Mean lag-1 autocorr  Min lag-1 autocorr  Max lag-1 autocorr  % subjects Ljung-Box sig (p<.05)  n subjects tested
    TIB                   1.295               -0.001              -0.511               0.465                             5.263                 38
   WASO                   1.314               -0.024              -0.368               0.652                             2.632                 38
    TST                   1.204               -0.047              -0.769               0.510                             7.895                 38
     SE                   1.226               -0.055              -0.622               0.669                             2.632                 38
    SOL                   1.855               -0.072              -0.429               0.314                             0.000                 38



RESIDUAL DIAGNOSTICS — Relative Humidity
— Residual normality by outcome, Relative Humidity (worst first) —
Outcome   n  Shapiro W  Shapiro p   Skew  Excess kurtosis
    SOL 694      0.840      0.000  1.697            3.025
   WASO 694      0.986      0.000  0.447            0.139
    TIB 694      0.991      0.000 -0.195            0.688
     SE 694      0.995      0.013 -0.208           -0.230
    TST 694      0.997      0.220 -0.043            0.326


...


— Autocorrelation across consecutive nights, Wet Bulb Temperature, by outcome —
Outcome  Durbin-Watson (global)  Mean lag-1 autocorr  Min lag-1 autocorr  Max lag-1 autocorr  % subjects Ljung-Box sig (p<.05)  n subjects tested
    TIB                   1.291                0.002              -0.505               0.461                             5.263                 38
   WASO                   1.309               -0.023              -0.352               0.724                             2.632                 38
    TST                   1.192               -0.042              -0.756               0.479                             5.263                 38
     SE                   1.219               -0.057              -0.626               0.758                             7.895                 38
    SOL                   1.854               -0.076              -0.429               0.329                             0.000                 38



RESIDUAL DIAGNOSTICS — CO2
— Residual normality by outcome, CO2 (worst first) —
Outcome   n  Shapiro W  Shapiro p   Skew  Excess kurtosis
    SOL 694      0.840      0.000  1.695            3.011
   WASO 694      0.986      0.000  0.457            0.194
    TIB 694      0.992      0.001 -0.167            0.715
     SE 694      0.995      0.021 -0.206           -0.226
    TST 694      0.997      0.233 -0.042            0.325



— Autocorrelation across consecutive nights, CO2, by outcome —
Outcome  Durbin-Watson (global)  Mean lag-1 autocorr  Min lag-1 autocorr  Max lag-1 autocorr  % subjects Ljung-Box sig (p<.05)  n subjects tested
    TIB                   1.290               -0.001              -0.503               0.465                             5.263                 38
   WASO                   1.321               -0.033              -0.443               0.472                             5.263                 38
    TST                   1.225               -0.055              -0.803               0.530                             7.895                 38
     SE                   1.239               -0.060              -0.672               0.417                            10.526                 38
    SOL                   1.848               -0.074              -0.433               0.346                             0.000                 38



RESIDUAL DIAGNOSTICS — PM2.5
— Residual normality by outcome, PM2.5 (worst first) —
Outcome   n  Shapiro W  Shapiro p   Skew  Excess kurtosis
    SOL 694      0.840      0.000  1.703            3.072
   WASO 694      0.986      0.000  0.457            0.184
    TIB 694      0.991      0.000 -0.178            0.725
     SE 694      0.995      0.021 -0.210           -0.213
    TST 694      0.997      0.217 -0.042            0.331



— Autocorrelation across consecutive nights, PM2.5, by outcome —
Outcome  Durbin-Watson (global)  Mean lag-1 autocorr  Min lag-1 autocorr  Max lag-1 autocorr  % subjects Ljung-Box sig (p<.05)  n subjects tested
    TIB                   1.298               -0.001              -0.512               0.468                             5.263                 38
   WASO                   1.323               -0.030              -0.391               0.481                             2.632                 38
    TST                   1.227               -0.056              -0.797               0.533                            10.526                 38
     SE                   1.240               -0.059              -0.634               0.412                            10.526                 38
    SOL                   1.848               -0.076              -0.430               0.360                             0.000                 38


## 10. Forest Plot (Figure 2)

In [23]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

FONT_SIZE = 24
SIG_ALPHA = 0.05


MARKER_SIZE = 18

# ---- Data ------------------------------------------------------------------
# raw (beta, ci_lo, ci_hi, adjusted_p) exactly as reported in the source table.
# CO2 is per 100 ppm increment; PM2.5 is per 10 ug/m3 increment.
ENV_ORDER = [
    "Temperature",
    "Absolute Humidity",
    "Relative Humidity",
    "Wet Bulb Temperature",
    "CO2",
    "PM2.5",
]

ENV_LABELS = {
    "Temperature": "Temperature (1 °C ↑)",
    "Absolute Humidity": "Absolute Humidity (1 g/m³ ↑)",
    "Relative Humidity": "Relative Humidity (1% ↑)",
    "Wet Bulb Temperature": "Wet Bulb Temperature (1 °C ↑)",
    "CO2": "CO₂ (100 ppm ↑)",
    "PM2.5": "PM₂.₅ (10 μg/m³ ↑)",
}

ENV_DATA = {
    exposure: {
        outcome: (row["beta"], row["ci_lo"], row["ci_hi"], row["p_bonferroni"])
        for outcome, row in ols_results[exposure]["summary"].iterrows()
    }
    for exposure in ENV_ORDER
}

panels = [
    dict(metric="TIB", panel_type="decrease", x_range=[-18, 11], x_title="Change in Time In Bed (min)"),
    dict(metric="TST", panel_type="decrease", x_range=[-18, 11], x_title="Change in Total Sleep Time (min)"),
    dict(metric="SE", panel_type="decrease", x_range=[-3, 1.4], x_title="Change in Sleep Efficiency (%)"),
    dict(metric="SOL", panel_type="increase", x_range=[-3.6, 2.4], x_title="Change in Sleep Onset Latency (min)"),
    dict(metric="WASO", panel_type="increase", x_range=[-7, 12], x_title="Change in Wake After Sleep Onset (min)"),
]


# ---- Grid layout -------------------------------------------------------------
# 2 rows x 6 columns. Row 1 holds 3 plots, each spanning 2 columns (cols 1-2,
# 3-4, 5-6). Row 2 holds 2 plots, each spanning 2 columns, centered by leaving
# a 1-column pad on either side (pad, 2-3, 4-5, pad).
specs = [
    [{"colspan": 2}, None, {"colspan": 2}, None, {"colspan": 2}, None],
    [None, {"colspan": 2}, None, {"colspan": 2}, None, None],
]
GRID_POSITIONS = [(1, 1), (1, 3), (1, 5), (2, 2), (2, 4)]
PANEL_LABELS = ["(a)", "(b)", "(c)", "(d)", "(e)"]

# ---- Figure ------------------------------------------------------------------
fig = make_subplots(
    rows=2,
    cols=6,
    specs=specs,
    horizontal_spacing=0.175,
    vertical_spacing=0.16,
)

for (row, col), panel in zip(GRID_POSITIONS, panels):
    metric = panel["metric"]
    panel_lo, panel_hi = [], []

    for env in ENV_ORDER:
        beta, ci_lo, ci_hi, p = ENV_DATA[env][metric]

        plotted_beta = beta
        plotted_lo = ci_lo
        plotted_hi = ci_hi
        panel_lo.append(plotted_lo)
        panel_hi.append(plotted_hi)

        significant = p < SIG_ALPHA
        marker = dict(
            symbol="diamond" if significant else "circle",
            size=MARKER_SIZE + 2 if significant else MARKER_SIZE,
            color="#D62728" if significant else "black",
        )

        fig.add_trace(
            go.Scatter(
                x=[plotted_beta],
                y=[ENV_LABELS[env]],
                mode="markers+text",
                marker=marker,
                cliponaxis=False,
                text=[f"{plotted_beta:+.1f}"],
                textposition="top center",
                textfont=dict(family="Arial", color="black", size=FONT_SIZE - 2),
                error_x=dict(
                    type="data",
                    symmetric=False,
                    array=[plotted_hi - plotted_beta],
                    arrayminus=[plotted_beta - plotted_lo],
                    color="black",
                    thickness=2,
                    width=6,
                ),
                name=env,
                legendgroup=env,
                showlegend=False,
            ),
            row=row, col=col,
        )

    fig.add_vline(x=0, line=dict(color="gray", width=1, dash="dash"), row=row, col=col)

    x_lo, x_hi = panel["x_range"]
    assert min(panel_lo) >= x_lo and max(panel_hi) <= x_hi, (
        f"\u274c {metric} panel's hard-coded x_range {panel['x_range']} does not "
        f"cover the plotted CIs [{min(panel_lo):.2f}, {max(panel_hi):.2f}] -- widen "
        f"x_range for this panel."
    )

    fig.update_xaxes(range=panel["x_range"], title_text=panel["x_title"], row=row, col=col)

    n = len(ENV_ORDER)
    fig.update_yaxes(
        categoryorder="array",
        categoryarray=[ENV_LABELS[e] for e in ENV_ORDER][::-1],
        range=[-0.4, n - 1 + 0.9],
        row=row, col=col,
    )

# ---- Panel labels (a)-(e): bold, top-left of each subplot, same font --------
# Subplot axes are numbered in creation order (1 = no suffix, then 2, 3, ...),
# which matches GRID_POSITIONS / panels order above.
# for i, label in enumerate(PANEL_LABELS):
#     suffix = "" if i == 0 else str(i + 1)
#     fig.add_annotation(
#         text=f"<b>{label}</b>",
#         xref=f"x{suffix} domain",
#         yref=f"y{suffix} domain",
#         x=-0.06,
#         y=1.12,
#         xanchor="left",
#         yanchor="top",
#         showarrow=False,
#         font=dict(family="Arial", color="black", size=FONT_SIZE),
#     )

# ---- Global styling: Arial, black text --------------------------------------
fig.update_layout(
    font=dict(family="Arial", color="black", size=FONT_SIZE),
    plot_bgcolor="white",
    paper_bgcolor="white",
    showlegend=False,
    margin=dict(t=100, l=200, r=40, b=80),
    height=1300,
    width=2500,
)

fig.update_xaxes(
    showline=True, linecolor="black", linewidth=2,
    ticks="outside", tickcolor="black",
    zeroline=False,
    tickfont=dict(family="Arial", color="black"),
    title_font=dict(family="Arial", color="black", size=FONT_SIZE),
)
fig.update_yaxes(
    showline=True, linecolor="black", linewidth=2,
    ticks="outside", tickcolor="black",
    tickfont=dict(family="Arial", color="black"),
    title_font=dict(family="Arial", color="black", size=FONT_SIZE),
)

fig.show()
fig.write_image(os.path.join(FIGURES_DIR, "forest_plot_all.png"), width=2100, height=1300, scale=1)
# fig.write_image(os.path.join(FIGURES_DIR, "forest_plot_all.tiff"), width=2100, height=1300, scale=6, format="tiff")  # 600 dpi TIFF (uncomment to export)
print("done")

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/plotly/io/_kaleido.py:510: DeprecationWarning:


Support for the 'engine' argument is deprecated and will be removed after September 2025.
Kaleido will be the only supported engine at that time.




done


## 11. Thermal Comfort & Preference (Figure 3)

Ratings of thermal sensation, thermal preference and air movement preference, reported before sleep (n=694).

In [24]:
import pandas as pd
import plotly.graph_objects as go

TITLE_SIZE = 30
LABEL_SIZE = 28
PCT_SIZE = 28
LEGEND_SIZE = 22
MIN_WIDTH_FOR_LABEL = 0.08  # 8%

BAR_AREA_WIDTH = 900
BAR_AREA_HEIGHT = 160
LEGEND_AREA_HEIGHT = 220
LEFT_MARGIN, RIGHT_MARGIN, TOP_MARGIN, BOTTOM_MARGIN = 20, 40, 80, 0


def report_category_percentages(df_in, col, categories, title):
    counts = df_in[[col]].dropna()[col].value_counts().reindex(categories).fillna(0)
    total = counts.sum()
    print("\n" + "=" * 70)
    print(title)
    print("=" * 70)
    for cat in categories:
        n = int(counts[cat])
        pct = (n / total * 100) if total > 0 else 0
        print(f"{cat:<18}: {n:>4} ({pct:5.1f}%)")
    print(f"Total N = {int(total)}")


def plot_horizontal_stacked_bar(df_in, col, title, categories, colors, bar_thickness=0.40):
    """One horizontal 100%-stacked bar showing the distribution of `col` over `categories`."""
    report_category_percentages(df_in, col, categories, title)

    counts = df_in[[col]].dropna()[col].value_counts().reindex(categories).fillna(0)
    total = counts.sum()

    fig_width = LEFT_MARGIN + BAR_AREA_WIDTH + RIGHT_MARGIN
    fig_height = TOP_MARGIN + max(BAR_AREA_HEIGHT, LEGEND_AREA_HEIGHT) + BOTTOM_MARGIN
    x_domain_end = BAR_AREA_WIDTH / (fig_width - LEFT_MARGIN - RIGHT_MARGIN)

    fig = go.Figure()
    if total > 0:
        props = counts / total
        for cat in categories:
            val = props[cat]
            if val <= 0:
                continue
            text_label = f"{val*100:.0f}%" if val >= MIN_WIDTH_FOR_LABEL else ""
            fig.add_trace(go.Bar(
                y=[""], x=[val], name=cat, orientation="h", width=bar_thickness,
                marker=dict(color=colors[cat]), text=[text_label], textposition="inside",
                textfont=dict(size=PCT_SIZE, color="black"), insidetextanchor="middle",
                hovertemplate=f"{cat}<br>Count: {int(counts[cat])}<br>Percent: {val*100:.1f}%<extra></extra>",
            ))

    fig.update_layout(
        autosize=False, width=fig_width, height=fig_height, barmode="stack",
        plot_bgcolor="white", paper_bgcolor="white",
        font=dict(family="Arial", size=LABEL_SIZE, color="black"),
        title=dict(text=f"<b>{title}</b>", x=0.4, y=0.9, xanchor="center", font=dict(size=TITLE_SIZE, color="black")),
        margin=dict(l=LEFT_MARGIN, r=RIGHT_MARGIN, t=TOP_MARGIN, b=BOTTOM_MARGIN),
        legend=dict(traceorder="normal", orientation="h", x=0.5, xanchor="center", y=0.74, yanchor="bottom",
                    font=dict(size=LEGEND_SIZE, color="black"), bgcolor="rgba(255,255,255,0)"),
        xaxis=dict(range=[0, 1], domain=[0, x_domain_end], showgrid=False, showticklabels=False,
                   zeroline=False, fixedrange=True),
        yaxis=dict(domain=[0, 1], showgrid=False, showticklabels=False, zeroline=False, fixedrange=True),
    )
    fig.show()
    # fig.write_image(os.path.join(FIGURES_DIR, title.replace(" ", "_") + ".tiff"), width=fig_width, height=fig_height, scale=6, format="tiff")  # 600 dpi TIFF (uncomment to export)
    # No return value -- see the note at the end of plot_profile() above.

### 11.1 Thermal Sensation (Figure 3a)

In [25]:
df = df_clean.copy()
for col in ["Q1.3 PRE-RTS"]:
    df[col] = df[col].astype(str).str.strip()

cats_rts = ["Cold", "Cool", "Slightly cool", "Neutral", "Slightly warm", "Warm", "Hot"]
cols_rts = {
    "Hot": "#a7514e", "Warm": "#db735f", "Slightly warm": "#ecadb7", "Neutral": "#cee0b5",
    "Slightly cool": "#c1e3f5", "Cool": "#91c4e9", "Cold": "#728cd0",
}

plot_horizontal_stacked_bar(df, "Q1.3 PRE-RTS", "BEF-Sleep Ratings of Thermal Sensation", cats_rts, cols_rts)


BEF-Sleep Ratings of Thermal Sensation
Cold              :   28 (  4.0%)
Cool              :   39 (  5.6%)
Slightly cool     :  104 ( 15.0%)
Neutral           :  205 ( 29.5%)
Slightly warm     :  172 ( 24.8%)
Warm              :   33 (  4.8%)
Hot               :  113 ( 16.3%)
Total N = 694


### 11.2 Thermal Preference (Figure 3b)

In [26]:
df = df_clean.copy()
for col in ["Q1.4 RTP"]:
    df[col] = df[col].astype(str).str.strip()

cats_rtp = ["Warmer", "Without change", "Cooler"]
cols_rtp = {"Cooler": "#db735f", "Without change": "#cce1b6", "Warmer": "#91c4e9"}

plot_horizontal_stacked_bar(df, "Q1.4 RTP", "BEF-Sleep Ratings of Thermal Preference", cats_rtp, cols_rtp)


BEF-Sleep Ratings of Thermal Preference
Warmer            :    3 (  0.4%)
Without change    :  169 ( 24.4%)
Cooler            :  522 ( 75.2%)
Total N = 694


### 11.3 Air Movement Preference (Figure 3c)

In [27]:
df = df_clean.copy()
label_map = {"More wind": "More air movement", "No change": "No change", "Less wind": "Less air movement"}
for col in ["Q1.5 Wind"]:
    df[col] = df[col].astype(str).str.strip().replace(label_map)

cats_amp = ["Less air movement", "No change", "More air movement"]
cols_amp = {"More air movement": "#db735f", "No change": "#cce1b6", "Less air movement": "#91c4e9"}

plot_horizontal_stacked_bar(df, "Q1.5 Wind", "BEF-Sleep Ratings of Air Movement Preference", cats_amp, cols_amp)


BEF-Sleep Ratings of Air Movement Preference
Less air movement :   25 (  3.6%)
No change         :  146 ( 21.0%)
More air movement :  523 ( 75.4%)
Total N = 694
